In [5]:
# import all the necessary modules
import os
import copy
import json
import config
import code_execution
import RAG.code_reference
import RAG.model_information
import RAG.model_optimization
import cross_platform_protocol
import config, prompt, agent, tool_list

from langchain_community.document_loaders.firecrawl import FireCrawlLoader

In [2]:
def get_model_deployment_code(model_url, model_name):
    # retrieve and save the model information into the database
    web_loader = FireCrawlLoader(
        api_key=os.environ["FIRECRAWL_API_KEY"], url=model_url, mode="scrape"
    )
    model_information_document = []
    for document in web_loader.lazy_load():
        model_information_document.append(document)

    document_list = tool_list.text_splitter.split_documents(model_information_document)

    for index in range(len(document_list)):
        current_document = document_list[index]
        current_metadata = current_document.metadata

        for key, value in current_metadata.items():
            if isinstance(value, list):
                document_list[index].metadata[key] = str(value)

    RAG.model_information.model_information_database.add_documents(
        documents=document_list
    )

    chat_history = RAG.model_information.model_information_graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt.model_information_retrieval_prompt.format(
                        model_name
                    ),
                }
            ]
        }
    )
    model_description = chat_history["messages"][-1].content
    model_deployment_code = tool_list.sanitize_code(model_description)

    return model_deployment_code, model_name

In [3]:
def get_code_reference(mode: bool, user_requirements: str):
    # mode = True: retrieve the reference code for resource monitoring
    # mode = False: retrieve the reference code for performance monitoring
    chat_history = RAG.code_reference.code_reference_graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": (
                        prompt.resource_monitoring_code_reference_prompt.format(
                            user_requirements
                        )
                        if mode
                        else prompt.performance_monitoring_code_reference_prompt.format(
                            user_requirements
                        )
                    ),
                }
            ]
        }
    )
    reference_code = chat_history["messages"][-1].content.strip()

    return reference_code

In [5]:
# =================== MODULE 1 ===================
# Module 1: Requirement Standardization via User-in-the-loop
print("Please enter the model name you would like to deploy:")
model_name = input()
# model_name = "ResNet-152"
print("Please enter the URL of the corresponding Hugging Face model:")
model_url = input()
# model_url = "https://huggingface.co/microsoft/resnet-152"
print("Now, please enter the target edge device you would like to deploy the model on:")
device_name = input()
# device_name = "Raspberry Pi 5"

# the agent search the device name within the Internet and provide detailed information about the device to the user
print("Entering device specs searching chain . . .")
device_specs = agent.device_information_search_chain.invoke(device_name)
device_specs = device_specs.content
print("\n\n{}".format(device_specs))

Please enter the model name you would like to deploy:
Please enter the URL of the corresponding Hugging Face model:
Now, please enter the target edge device you would like to deploy the model on:
Entering device specs searching chain . . .
We have searched the following information:
url: https://community.element14.com/products/raspberry-pi/b/blog/posts/raspberry-pi-5-technical-specifications-and-mechanical-drawings
content: Raspberry Pi 5 Key Features:Here are the tech specs for the brand new Raspberry Pi 5: • 2.4GHz quad-core, 64-bit Arm Cortex-A76 CPU, with 512KB L2 caches and a 2MB shared L3 cache • VideoCore VII GPU • LPDDR4X-4267 SDRAM (4GB and 8GB SKUs available at launch) • 2.4 GHz and 5.0 GHz 802.11ac
url: https://archive.org/download/raspberry-pi-5-technical-drawing-webpage/Raspberry+Pi+5+Technical+Drawing+Webpage.pdf
content: Raspberry Pi 5 Technical Specifications and Mechanical Drawings 26 sep 2023 Raspberry Pi 5 Key Features: Here are the tech specs for the brand new Rasp

In [ ]:
# next, the agent ask the user to provide her requirements following our template
print(
    "Now, please specify your requirements for the model to be deployed on {}".format(
        device_name
    )
)
print("Here is a list of resource related requirements you can specify:")
print("CPU memory, GPU memory, and local storage")
print("Here is a list of inference performance related requirements you can specify:")
print("Inference latency and throughput")
print("Here is a list of model accuracy related requirements you can specify:")
print("For classification models: accuracy, precision, recall rate, and F1-score.")
print("For regression models: MAE, MSE, RMSE, R-squared, and MAPE.")
print("For large language models: BLEU score and ROUGE score.")

print("Now, please input your requirements:")
requirements = input()
# requirements = "To deploy a ResNet-152 model on Raspberry Pi 5, the model's CPU memory usage should stay below 3120 MB. The model parameters should take up less than 200 MB of disk space. Inference latency must be ≤ 150 ms. The system should process at least 5 images per second while maintaining a top-1 accuracy of ≥ 65%, precision of ≥ 60%, recall of ≥ 55%, and F1-score of ≥ 60%. These requirements ensure efficient deployment with robust performance."

# process the raw natural language requirements into a structured format
standardized_requirement_generation_chain = (
    prompt.requirement_standardization_prompt | tool_list.model
)
standardized_requirements = standardized_requirement_generation_chain.invoke(
    requirements
)
standardized_requirements = standardized_requirements.content
print(standardized_requirements)

Now, please specify your requirements for the model to be deployed on Raspberry Pi 5
Here is a list of resource related requirements you can specify:
CPU memory, GPU memory, and local storage
Here is a list of inference performance related requirements you can specify:
Inference latency and throughput
Here is a list of model accuracy related requirements you can specify:
For classification models: accuracy, precision, recall rate, and F1-score.
For regression models: MAE, MSE, RMSE, R-squared, and MAPE.
For large language models: BLEU score and ROUGE score.
Now, please input your requirements:
- CPU memory usage: less than 3120 MB
- Model size: less than 200 MB of disk space
- Inference latency: less than or equal to 150 ms
- Throughput: at least 5 images per second
- Top-1 accuracy: greater than or equal to 75%
- Precision: greater than or equal to 80%
- Recall: greater than or equal to 85%
- F1-score: greater than or equal to 75%


In [37]:
# split the requirements into two parts: resource requirements and performance requirements
resource_requirement_extraction_chain = (
    prompt.resource_requirement_extraction_prompt | tool_list.model
)
resource_requirement = resource_requirement_extraction_chain.invoke(
    standardized_requirements
)
resource_requirement = resource_requirement.content.strip()
print("Resource related user requirements:\n{}".format(resource_requirement))
print()
print()

performance_requirement_extraction_chain = (
    prompt.performance_requirement_extraction_prompt | tool_list.model
)
performance_requirement = performance_requirement_extraction_chain.invoke(
    standardized_requirements
)
performance_requirement = performance_requirement.content.strip()
print("Performance related user requirements:\n{}".format(performance_requirement))

Resource related user requirements:
CPU memory usage: less than 3120 MB  
Model size: less than 200 MB of disk space


Performance related user requirements:
Inference latency: less than or equal to 150 ms  
Throughput: at least 5 images per second  
Top-1 accuracy: greater than or equal to 75%  
Precision: greater than or equal to 80%  
Recall: greater than or equal to 85%  
F1-score: greater than or equal to 75%


In [6]:
print("Do you require that the model will be executed in the GPU? Y/N")
gpu_required = input()
# gpu_required = "N"
gpu_required = True if gpu_required == "Y" else False
deployment_device = "GPU" if gpu_required else "CPU"

Do you require that the model will be executed in the GPU? Y/N


In [ ]:
print(
    "Now, please provide detailed information of your dataset residing on the edge device."
)
print(
    """The dataset information should include the following information:
- For classification tasks:
  1. Data characteristics: data type, dimension, and the meaning of each dimension.
  2. Classes and labels: the number of classes and class names.
  3. Dataset size: the number of samples in the dataset.
  4. Expected input and output format.
- For regression tasks:
  1. Data characteristics: data type, dimension, and the meaning of each dimension.
  2. Dataset size: the number of samples in the dataset.
  3. Expected input and output format.
- For next-token prediction tasks:
  1. Data characteristics: text languages, data dimension, and the meaning of each dimension.
  2. Vocabulary size: the number of unique tokens in the dataset.
  3. Dataset size: the number of samples in the dataset.
  4. Expected input and output format."""
)
dataset_information = input()
# dataset_information = """The dataset contains 60000 224x224 RGB images in 1000 classes. There are 50000 training images and 10000 testing images. Each sample is represented as a (3, 224, 224) numpy matrix. The test images are stored in a (10000, 3, 224, 224) numpy array and the test labels are stored in a (10000,) numpy array. Note that the data in the dataset can be directly fed into the model for inference without preprocessing (e.g., feature extraction)."""
local_dataset_path = "./dataset/ILSVRC"

Now, please provide detailed information of your dataset residing on the edge device.
The dataset information should include the following information:
- For classification tasks:
  1. Data characteristics: data type, dimension, and the meaning of each dimension.
  2. Classes and labels: the number of classes and class names.
  3. Dataset size: the number of samples in the dataset.
  4. Expected input and output format.
- For regression tasks:
  1. Data characteristics: data type, dimension, and the meaning of each dimension.
  2. Dataset size: the number of samples in the dataset.
  3. Expected input and output format.
- For next-token prediction tasks:
  1. Data characteristics: text languages, data dimension, and the meaning of each dimension.
  2. Vocabulary size: the number of unique tokens in the dataset.
  3. Dataset size: the number of samples in the dataset.
  4. Expected input and output format.


In [9]:
# generate code for local model deployment
model_deployment_code, model_name = get_model_deployment_code(model_url, model_name)
print()
print("\033[34m{}\033[0m".format(model_deployment_code))

Retrieved the following information from the database:
Source: {'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'fb:app_id': '1321688464574422', 'og:image': 'https://cdn-thumbnails.huggingface.co/social-thumbnails/models/microsoft/resnet-152.png', 'og:title': 'microsoft/resnet-152 · Hugging Face', 'og:type': 'website', 'og:url': 'https://huggingface.co/microsoft/resnet-152', 'ogImage': 'https://cdn-thumbnails.huggingface.co/social-thumbnails/models/microsoft/resnet-152.png', 'ogTitle': 'microsoft/resnet-152 · Hugging Face', 'ogUrl': 'https://huggingface.co/microsoft/resnet-152', 'scrapeId': '8fc499c5-6828-4db6-b5a7-b8fc516f2b9f', 'sourceURL': 'https://huggingface.co/microsoft/resnet-152', 'statusCode': 200, 'title': 'microsoft/resnet-152 · Hugging Face', 'twitter:card': 'summary_large_image', 'twitter:image': 'https://cdn-thumbnails.huggingface.co/social-thumbnails/models/microsoft/resnet-152.png', 'twitter:s

In [10]:
# =================== MODULE 2 ===================
# Module 2: Entangled model optimization and deployment
## Task 1: Model profiling via local deployment (DEPLOYMENT AGENT)
### Task 1.1: generate code for model running on local device
prompt.model_deployment_code_prompt["messages"][0]["content"] = (
    prompt.model_deployment_code_prompt["messages"][0]["content"]
).format(model_name, model_deployment_code)
prompt.model_deployment_code_prompt["messages"][1]["content"] = (
    prompt.model_deployment_code_prompt["messages"][1]["content"]
).format(deployment_device)
agent_output = agent.local_deployment_agent.invoke(
    input=prompt.model_deployment_code_prompt,
    config=agent.local_deployment_agent_configuration,
)
model_local_deployment_code = agent_output["messages"][-1].content
model_local_deployment_code = tool_list.sanitize_code(model_local_deployment_code)
print("\033[34m{}\033[0m".format(model_local_deployment_code))

from transformers import AutoFeatureExtractor, ResNetForImageClassification
import torch
from datasets import load_dataset
import multiprocessing

def model_inference():
    dataset = load_dataset("huggingface/cats-image")
    image = dataset["test"]["image"][0]

    feature_extractor = AutoFeatureExtractor.from_pretrained("microsoft/resnet-152")
    model = ResNetForImageClassification.from_pretrained("microsoft/resnet-152")

    inputs = feature_extractor(image, return_tensors="pt")

    with torch.no_grad():
        logits = model(**inputs).logits

    # model predicts one of the 1000 ImageNet classes
    predicted_label = logits.argmax(-1).item()
    print(model.config.id2label[predicted_label])

if __name__ == "__main__":
    p = multiprocessing.Process(target=model_inference)
    p.start()
    p.join()


In [11]:
# Task 1.2: Modify the code by adding local datasets
prompt.model_deployment_on_local_test_dataset_prompt["messages"][0]["content"] = (
    prompt.model_deployment_on_local_test_dataset_prompt["messages"][0][
        "content"
    ].format(local_dataset_path, dataset_information)
)
agent_output = agent.local_deployment_agent.invoke(
    input=prompt.model_deployment_on_local_test_dataset_prompt,
    config=agent.local_deployment_agent_configuration,
)
model_local_deployment_code = agent_output["messages"][-1].content
model_local_deployment_code = tool_list.sanitize_code(model_local_deployment_code)
base_deployment_code = copy.deepcopy(model_local_deployment_code)

print("Code for local deployment and monitoring with local dataset.")
print("\033[34m{}\033[0m".format(model_local_deployment_code))

Code for local deployment and monitoring with local dataset.
from transformers import ResNetForImageClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import multiprocessing

def model_inference():
    # Load the test data and labels from the local path
    test_data = np.load('./dataset/ILSVRC/test_data.npy')
    test_labels = np.load('./dataset/ILSVRC/test_label.npy')

    # Convert numpy arrays to PyTorch tensors
    test_data_tensor = torch.tensor(test_data, dtype=torch.float32)
    test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)

    # Pack the data and labels into a DataLoader object
    test_dataset = TensorDataset(test_data_tensor, test_labels_tensor)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Load the model
    model = ResNetForImageClassification.from_pretrained("microsoft/resnet-152")
    model.eval()

    correct = 0
    total = 0

    # Evaluate the model on the local d

In [12]:
### Task 1.3: Generate code for model monitoring on local device
#### Task 1.3.1: retrieve relevant reference code
resource_reference_code = get_code_reference(
    mode=True, user_requirements=resource_requirement
)

print("Retrieved reference code for resource monitoring:")
print("\033[34m{}\033[0m".format(resource_reference_code))

Retrieved reference code for resource monitoring:
1. Reference code for CPU memory monitoring
```python
import os
import time
import psutil
import threading
import multiprocessing

from tqdm import tqdm


def model_inference():
    finished = False
    CPU_memory_list = []
    handle = psutil.Process(os.getpid())

    # CPU memory monitoring function
    def monitor_resources():
        while not finished:
            time.sleep(1)
            cpu_memory = handle.memory_info().rss / 1024 / 1024  # in MB
            CPU_memory_list.append(cpu_memory)

    # Start the monitoring thread
    thread = threading.Thread(target=monitor_resources)
    thread.start()

    # create model, data loader, etc.
    # model = None
    # data_loader = None

    # You may deploy the model here
    # Model deployment, inference, or any other task ...
    # for data, label in tqdm(data_loader):
    # ......

    # Stop the monitoring thread when the work is done
    finished = True
    thread.join()
    ma

In [13]:
#### Task 1.3.2: generate code for model monitoring on local device
prompt.model_execution_monitoring_prompt["messages"][0]["content"] = (
    prompt.model_execution_monitoring_prompt["messages"][0]["content"].format(
        resource_reference_code
    )
)
model_local_deployment_code = agent.local_deployment_agent.invoke(
    input=prompt.model_execution_monitoring_prompt,
    config=agent.local_deployment_agent_configuration,
)
model_local_deployment_code = model_local_deployment_code["messages"][-1].content
model_local_deployment_code = tool_list.sanitize_code(model_local_deployment_code)
print("Code for local deployment and monitoring.")
print("\033[34m{}\033[0m".format(model_local_deployment_code))

Code for local deployment and monitoring.
from transformers import ResNetForImageClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
import time
import psutil
import threading
import multiprocessing

def model_inference():
    finished = False
    CPU_memory_list = []
    handle = psutil.Process(os.getpid())

    # CPU memory monitoring function
    def monitor_resources():
        while not finished:
            time.sleep(1)
            cpu_memory = handle.memory_info().rss / 1024 / 1024  # in MB
            CPU_memory_list.append(cpu_memory)

    # Start the monitoring thread
    thread = threading.Thread(target=monitor_resources)
    thread.start()

    # Load the test data and labels from the local path
    test_data = np.load('./dataset/ILSVRC/test_data.npy')
    test_labels = np.load('./dataset/ILSVRC/test_label.npy')

    # Convert numpy arrays to PyTorch tensors
    test_data_tensor = torch.tensor(test_data, dtype=tor

In [14]:
#### Task 1.3.3: modify the code by loading model parameters from local disk
prompt.model_inference_script_generation_prompt["messages"][0]["content"] = (
    prompt.model_inference_script_generation_prompt["messages"][0]["content"].format(
        model_name
    )
)
agent_output = agent.local_deployment_agent.invoke(
    input=prompt.model_inference_script_generation_prompt,
    config=agent.local_deployment_agent_configuration,
)
agent_output = agent_output["messages"][-1].content
model_local_deployment_code = tool_list.sanitize_code(agent_output)

print("Code for local deployment and monitoring.")
print("\033[34m{}\033[0m".format(model_local_deployment_code))

Code for local deployment and monitoring.
from transformers import ResNetForImageClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
import time
import psutil
import threading
import multiprocessing

def model_inference():
    finished = False
    CPU_memory_list = []
    handle = psutil.Process(os.getpid())

    # CPU memory monitoring function
    def monitor_resources():
        while not finished:
            time.sleep(1)
            cpu_memory = handle.memory_info().rss / 1024 / 1024  # in MB
            CPU_memory_list.append(cpu_memory)

    # Start the monitoring thread
    thread = threading.Thread(target=monitor_resources)
    thread.start()

    # Load the test data and labels from the local path
    test_data = np.load('./dataset/ILSVRC/test_data.npy')
    test_labels = np.load('./dataset/ILSVRC/test_label.npy')

    # Convert numpy arrays to PyTorch tensors
    test_data_tensor = torch.tensor(test_data, dtype=tor

In [16]:
# Task 1.3: Execute the model and record the monitoring results
with open("local_deployment.py", "w", encoding="utf-8") as file:
    file.write(model_local_deployment_code)
execution_command = "python local_deployment.py"
status, program_output = code_execution.run(execution_command, True)
print("The resource monitoring results of the original model:")
print("\033[34m{}\033[0m".format(program_output))

************** START **************
100%|██████████| 63/63 [03:27<00:00,  3.29s/it]
Accuracy: 83.35%
Maximum CPU memory usage: 3369.66015625 MB
Model size in local storage: 230.699556350708 MB
************** SUCCESS **************
The resource monitoring results of the original model:
100%|██████████| 63/63 [03:27<00:00,  3.29s/it]
Accuracy: 83.35%
Maximum CPU memory usage: 3369.66015625 MB
Model size in local storage: 230.699556350708 MB


In [17]:
# Task 1.4: Filter the program output and only keep the useful information
program_output_filter_chain = prompt.program_output_filtering_prompt | tool_list.model
filtered_program_output = program_output_filter_chain.invoke(program_output).content

print("The resource monitoring results of the original model after filtering:")
print("\033[34m{}\033[0m".format(filtered_program_output))

The resource monitoring results of the original model after filtering:
Accuracy: 83.35%  
Maximum CPU memory usage: 3369.66015625 MB  
Model size in local storage: 230.699556350708 MB


In [18]:
### Task 2.1: Generate different optimization methods
#### Task 2.1.1: Generate a list of potential hyperparameters that can be optimized
prompt.optimization_method_generation_prompt["messages"][1]["content"] = (
    prompt.optimization_method_generation_prompt["messages"][1]["content"].format(
        model_local_deployment_code, filtered_program_output
    )
)
agent_output = agent.optimization_method_generation_agent.invoke(
    input=prompt.optimization_method_generation_prompt,
    config=agent.optimization_method_generation_agent_configuration,
)
optimizable_hyperparameter_list = agent_output["messages"][-1].content

print(
    "The optimizable hyperparameters are:\n{}".format(optimizable_hyperparameter_list)
)

optimizable_hyperparameter_list = optimizable_hyperparameter_list.split("\n")

The optimizable hyperparameters are:
Hyperparameter name 1: model architecture (e.g., ResNet depth)
Hyperparameter name 2: batch size
Hyperparameter name 3: data precision (e.g., float32, float16)
Hyperparameter name 4: model pruning level
Hyperparameter name 5: quantization level (e.g., 8-bit, 16-bit)
Hyperparameter name 6: number of model layers to use
Hyperparameter name 7: number of output classes


In [19]:
### Task 2.1.2: Filter out those hyperparameters that are not suitable for optimization
filtered_hyperparameter_list = []
for optimizable_hyperparameter in optimizable_hyperparameter_list:
    hyperparameter_filtering_chain = (
        prompt.hyperparameter_filtering_prompt | tool_list.model
    )
    current_condition = hyperparameter_filtering_chain.invoke(
        input={"device": deployment_device, "input": optimizable_hyperparameter}
    )
    current_condition = current_condition.content.lower()

    if current_condition == "yes":
        filtered_hyperparameter_list.append(optimizable_hyperparameter)

print("The filtered optimizable hyperparameters are:\n")
for filtered_hyperparameter in filtered_hyperparameter_list:
    print(filtered_hyperparameter)

The filtered optimizable hyperparameters are:

Hyperparameter name 2: batch size
Hyperparameter name 3: data precision (e.g., float32, float16)
Hyperparameter name 4: model pruning level
Hyperparameter name 5: quantization level (e.g., 8-bit, 16-bit)


In [24]:
### Task 2.1.3: Analyze the hyperparameters of the original model
prompt.hyperparameter_analysis_prompt["messages"][0]["content"] = (
    prompt.hyperparameter_analysis_prompt["messages"][0]["content"].format(
        base_deployment_code, filtered_hyperparameter_list
    )
)

agent_output = agent.general_search_agent.invoke(
    input=prompt.hyperparameter_analysis_prompt
)
default_hyperparameter_configuration = agent_output["messages"][-1].content
print(default_hyperparameter_configuration)

* Batch size: 32
* Model pruning ratio: None
* Quantization level: None


In [26]:
### Task 2.2: Generate tuning code for each hyperparameter
tuning_code_list = []
for filtered_hyperparameter in filtered_hyperparameter_list:
    current_agent_configuration = {
        "configurable": {
            "thread_id": "hyperparameter_tuning_{}".format(filtered_hyperparameter)
        }
    }
    try:
        #### Task 2.2.1: Apply a random value of optimization hyperparameter to the model
        current_model_optimization_adoption_prompt = copy.deepcopy(
            prompt.model_optimization_adoption_prompt
        )
        current_model_optimization_adoption_prompt["messages"][0]["content"] = (
            current_model_optimization_adoption_prompt["messages"][0]["content"].format(
                base_deployment_code
            )
        )
        current_model_optimization_adoption_prompt["messages"][1][
            "content"
        ] = current_model_optimization_adoption_prompt["messages"][1]["content"].format(
            filtered_hyperparameter,
            filtered_hyperparameter,
            "GPU" if gpu_required else "CPU",
        )
        agent_output = agent.local_optimization_agent.invoke(
            input=current_model_optimization_adoption_prompt,
            config=current_agent_configuration,
        )

        #### Task 2.2.2: Generate search space for each hyperparameter
        current_search_space_generation_prompt = copy.deepcopy(
            prompt.search_space_generation_prompt
        )
        current_search_space_generation_prompt["messages"][0]["content"] = (
            current_search_space_generation_prompt["messages"][0]["content"].format(
                filtered_hyperparameter
            )
        )
        agent_output = agent.local_optimization_agent.invoke(
            input=current_search_space_generation_prompt,
            config=current_agent_configuration,
        )

        #### Task 2.2.3: Modify the objective function based on user requirements
        current_objective_function_modification_prompt = copy.deepcopy(
            prompt.objective_function_modification_prompt
        )
        current_objective_function_modification_prompt["messages"][0]["content"] = (
            current_objective_function_modification_prompt["messages"][0][
                "content"
            ].format(resource_requirement, resource_reference_code)
        )
        agent_output = agent.local_optimization_agent.invoke(
            input=current_objective_function_modification_prompt,
            config=current_agent_configuration,
        )

        #### Task 2.2.4: Modify the search scope and print out the optimal hyperparameter
        current_search_scope_modification_prompt = copy.deepcopy(
            prompt.search_scope_modification_prompt
        )
        current_search_scope_modification_prompt["messages"][0][
            "content"
        ] = current_search_scope_modification_prompt["messages"][0]["content"].format(
            filtered_hyperparameter,
            resource_requirement,
            default_hyperparameter_configuration,
            filtered_program_output,
        )
        agent_output = agent.local_optimization_agent.invoke(
            input=current_search_scope_modification_prompt,
            config=current_agent_configuration,
        )

        #### Task 2.2.5: Modify the code that output all possible hyperparameters
        agent_output = agent.local_optimization_agent.invoke(
            input=prompt.output_all_possible_hyperparameters_prompt,
            config=current_agent_configuration,
        )
        agent_output = agent_output["messages"][-1].content
        agent_output = tool_list.sanitize_code(agent_output)
        print("Sample code for tuning {}:".format(filtered_hyperparameter))
        tuning_code_list.append(agent_output)
        print("\033[34m{}\033[0m".format(agent_output))
        print("=========================================")
    except:
        print(
            "Error occurred during the tuning code generation for {}.".format(
                filtered_hyperparameter
            )
        )
        if filtered_hyperparameter in tool_list.agent_memory.storage.keys():
            tool_list.agent_memory.storage[filtered_hyperparameter].clear()

Number of requested results 5 is greater than number of elements in index 2, updating n_results = 2


Retrieved the following information from the database:
[Document(id='0dd8aca8-a943-4f13-85bb-cd94c6caa994', metadata={'description': 'Sample code for model quantization using PyTorch'}, page_content='import torch\nimport torch.ao.quantization\nfrom torchvision.models import resnet18\nfrom tqdm import tqdm\n\nimport time\nimport os\nimport numpy as np\nfrom torch.utils.data import DataLoader, TensorDataset\n\n# Load the test data and labels\ntest_data = np.load("./dataset/ILSVRC/test_data.npy")\ntest_labels = np.load("./dataset/ILSVRC/test_label.npy")\n\n# Convert to PyTorch tensors\ntest_data_tensor = torch.tensor(test_data, dtype=torch.float32)\ntest_labels_tensor = torch.tensor(test_labels, dtype=torch.long)\n\n# Create a DataLoader\ntest_dataset = TensorDataset(test_data_tensor, test_labels_tensor)\ntest_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)\n\nmodel = resnet18()\nmodel.eval()\n\n# Perform model quantization\n# Only qint8 is supported for dynamic quantizati

Number of requested results 5 is greater than number of elements in index 2, updating n_results = 2


Retrieved the following information from the database:
[Document(id='a27ea48b-b7a6-428c-b023-567017ff0940', metadata={'description': 'Sample code for model pruning using `torch_pruning` for PyTorch models.', 'source': 'https://github.com/VainF/Torch-Pruning'}, page_content='import torch\nfrom torchvision.models import resnet18\nimport torch_pruning as tp\n\nmodel = resnet18()\nclass_number = 1000\n# example input for the model\nexample_inputs = torch.randn(1, 3, 224, 224)  # (batch_size, channels, height, width)\n\n# 1. Importance criterion, here we calculate the L2 Norm of grouped weights as the importance score\nimp = tp.importance.GroupNormImportance(p=2)\n\n# 2. Initialize a pruner with the model and the importance criterion\nignored_layers = []\nfor m in model.modules():\n    if isinstance(m, torch.nn.Linear) and m.out_features == class_number:\n        ignored_layers.append(m)  # DO NOT prune the final classifier!\n\npruner = tp.pruner.MetaPruner(\n    model,\n    example_inputs,

Number of requested results 5 is greater than number of elements in index 2, updating n_results = 2


Retrieved the following information from the database:
[Document(id='a27ea48b-b7a6-428c-b023-567017ff0940', metadata={'description': 'Sample code for model pruning using `torch_pruning` for PyTorch models.', 'source': 'https://github.com/VainF/Torch-Pruning'}, page_content='import torch\nfrom torchvision.models import resnet18\nimport torch_pruning as tp\n\nmodel = resnet18()\nclass_number = 1000\n# example input for the model\nexample_inputs = torch.randn(1, 3, 224, 224)  # (batch_size, channels, height, width)\n\n# 1. Importance criterion, here we calculate the L2 Norm of grouped weights as the importance score\nimp = tp.importance.GroupNormImportance(p=2)\n\n# 2. Initialize a pruner with the model and the importance criterion\nignored_layers = []\nfor m in model.modules():\n    if isinstance(m, torch.nn.Linear) and m.out_features == class_number:\n        ignored_layers.append(m)  # DO NOT prune the final classifier!\n\npruner = tp.pruner.MetaPruner(\n    model,\n    example_inputs,

Number of requested results 5 is greater than number of elements in index 2, updating n_results = 2


Retrieved the following information from the database:
[Document(id='0dd8aca8-a943-4f13-85bb-cd94c6caa994', metadata={'description': 'Sample code for model quantization using PyTorch'}, page_content='import torch\nimport torch.ao.quantization\nfrom torchvision.models import resnet18\nfrom tqdm import tqdm\n\nimport time\nimport os\nimport numpy as np\nfrom torch.utils.data import DataLoader, TensorDataset\n\n# Load the test data and labels\ntest_data = np.load("./dataset/ILSVRC/test_data.npy")\ntest_labels = np.load("./dataset/ILSVRC/test_label.npy")\n\n# Convert to PyTorch tensors\ntest_data_tensor = torch.tensor(test_data, dtype=torch.float32)\ntest_labels_tensor = torch.tensor(test_labels, dtype=torch.long)\n\n# Create a DataLoader\ntest_dataset = TensorDataset(test_data_tensor, test_labels_tensor)\ntest_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)\n\nmodel = resnet18()\nmodel.eval()\n\n# Perform model quantization\n# Only qint8 is supported for dynamic quantizati

Number of requested results 5 is greater than number of elements in index 2, updating n_results = 2


Retrieved the following information from the database:
[Document(id='0dd8aca8-a943-4f13-85bb-cd94c6caa994', metadata={'description': 'Sample code for model quantization using PyTorch'}, page_content='import torch\nimport torch.ao.quantization\nfrom torchvision.models import resnet18\nfrom tqdm import tqdm\n\nimport time\nimport os\nimport numpy as np\nfrom torch.utils.data import DataLoader, TensorDataset\n\n# Load the test data and labels\ntest_data = np.load("./dataset/ILSVRC/test_data.npy")\ntest_labels = np.load("./dataset/ILSVRC/test_label.npy")\n\n# Convert to PyTorch tensors\ntest_data_tensor = torch.tensor(test_data, dtype=torch.float32)\ntest_labels_tensor = torch.tensor(test_labels, dtype=torch.long)\n\n# Create a DataLoader\ntest_dataset = TensorDataset(test_data_tensor, test_labels_tensor)\ntest_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)\n\nmodel = resnet18()\nmodel.eval()\n\n# Perform model quantization\n# Only qint8 is supported for dynamic quantizati

In [34]:
## Task 3: Model optimization code generation by applying various hyperparameter configurations
current_agent_configuration = {
    "configurable": {"thread_id": "search_space_combination"}
}

for index in range(len(tuning_code_list)):
    current_tuning_code = tuning_code_list[index]
    current_hyperparameter = filtered_hyperparameter_list[index]

    if index == 0:
        ### Task 3.1: Combine all the search spaces
        current_search_scope_combination_prompt = copy.deepcopy(
            prompt.search_scope_combination_prompt
        )

        current_search_scope_combination_prompt["messages"][0]["content"] = (
            current_search_scope_combination_prompt["messages"][0]["content"].format(
                base_deployment_code
            )
        )
        current_search_scope_combination_prompt["messages"][1][
            "content"
        ] = current_search_scope_combination_prompt["messages"][1]["content"].format(
            current_hyperparameter,
            current_hyperparameter,
            current_tuning_code,
            current_hyperparameter,
        )

        agent_output = agent.local_optimization_agent.invoke(
            input=current_search_scope_combination_prompt,
            config=current_agent_configuration,
        )
        agent_output = agent_output["messages"][-1].content
        print(agent_output)
    else:
        ### Task 3.2: Add more hyperparameters one by one
        current_search_scope_addition_prompt = copy.deepcopy(
            prompt.search_scope_addition_prompt
        )
        current_search_scope_addition_prompt["messages"][0][
            "content"
        ] = current_search_scope_addition_prompt["messages"][0]["content"].format(
            current_hyperparameter, current_tuning_code, current_hyperparameter
        )

        agent_output = agent.local_optimization_agent.invoke(
            input=current_search_scope_addition_prompt,
            config=current_agent_configuration,
        )
        agent_output = agent_output["messages"][-1].content
        print(agent_output)

```python
import optuna
from transformers import ResNetForImageClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import time
import os
import psutil
import threading
import multiprocessing

def evaluate_batch_size(batch_size):
    # Load the test data and labels from the local path
    test_data = np.load('./dataset/ILSVRC/test_data.npy')
    test_labels = np.load('./dataset/ILSVRC/test_label.npy')

    # Convert numpy arrays to PyTorch tensors
    test_data_tensor = torch.tensor(test_data, dtype=torch.float32)
    test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)

    # Pack the data and labels into a DataLoader object
    test_dataset = TensorDataset(test_data_tensor, test_labels_tensor)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Load the model
    model = ResNetForImageClassification.from_pretrained("microsoft/resnet-152")
    model.eval()

    # Save model to disk and mea

In [35]:
### Task 3.3: Adopt more advanced search strategies
agent_output = agent.local_optimization_agent.invoke(
    input=prompt.advanced_search_strategy_prompt,
    config=current_agent_configuration,
)
agent_output = agent_output["messages"][-1].content
local_optimization_code = tool_list.sanitize_code(agent_output)

print("The final code for model optimization:")
print("\033[34m{}\033[0m".format(local_optimization_code))

The final code for model optimization:
from transformers import ResNetForImageClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import torch.ao.quantization
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import SuccessiveHalvingPruner
import os
import psutil
import threading
import time
import torch_pruning as tp
import multiprocessing

def evaluate_model(batch_size, pruning_ratio, dtype):
    # Load the test data and labels from the local path
    test_data = np.load('./dataset/ILSVRC/test_data.npy')
    test_labels = np.load('./dataset/ILSVRC/test_label.npy')

    # Convert numpy arrays to PyTorch tensors
    test_data_tensor = torch.tensor(test_data, dtype=torch.float32)
    test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)

    # Pack the data and labels into a DataLoader object
    test_dataset = TensorDataset(test_data_tensor, test_labels_tensor)
    test_loader = DataLoader(test_dataset,

In [49]:
### Task 3.4: Execute the hyperparameter searching code and output all possible hyperparameter configurations
with open("local_optimization.py", "w", encoding="utf-8") as file:
    file.write(local_optimization_code)
execution_command = "python local_optimization.py"
status, program_output = code_execution.run(execution_command, True)
print("The resource monitoring results of the original model:")
print("\033[34m{}\033[0m".format(program_output))
program_output = program_output.split("\n")

************** START **************
[I 2025-02-17 11:06:02,858] A new study created in memory with name: no-name-516c1082-86b6-42ef-8fb3-9fba2dc93ebb
100%|██████████| 125/125 [01:23<00:00,  1.50it/s]
[I 2025-02-17 11:07:32,924] Trial 0 finished with values: [74.51937294006348, 3024.53125] and parameters: {'batch_size': 16, 'pruning_ratio': 0.40736660487537274, 'dtype': True}.
100%|██████████| 250/250 [01:11<00:00,  3.51it/s]
[I 2025-02-17 11:08:50,461] Trial 1 finished with values: [64.32827186584473, 2929.96875] and parameters: {'batch_size': 8, 'pruning_ratio': 0.4664172302277461, 'dtype': True}.
100%|██████████| 125/125 [01:23<00:00,  1.49it/s]
[I 2025-02-17 11:10:21,115] Trial 2 finished with values: [74.51937294006348, 3024.1640625] and parameters: {'batch_size': 16, 'pruning_ratio': 0.40995283896383095, 'dtype': True}.
100%|██████████| 250/250 [01:25<00:00,  2.93it/s]
[I 2025-02-17 11:11:52,588] Trial 3 finished with values: [86.02181434631348, 2861.59765625] and parameters: {'ba

In [ ]:
### Task 3.5: Parse the output of the running result
hyperparameter_configuration_list = []
start_index = 0
for index in range(len(program_output)):
    if (
        "100%" not in program_output[index]
        and "finished with values:" not in program_output[index]
        and "A new study created" not in program_output[index]
    ):
        start_index = index
        break

# program_output = program_output[start_index:]
for index in range(len(program_output) // 2):
    current_hyperparameter_configuration = program_output[index * 2 + 1]
    current_hyperparameter_configuration = current_hyperparameter_configuration[
        current_hyperparameter_configuration.find("{") :
    ]
    current_hyperparameter_configuration = (
        current_hyperparameter_configuration.replace("'", '"')
        .replace("False", "false")
        .replace("True", "true")
        .replace("dtype", "quantization")
    )
    current_hyperparameter_configuration = json.loads(
        current_hyperparameter_configuration
    )
    hyperparameter_configuration_list.append(current_hyperparameter_configuration)

print(hyperparameter_configuration_list)

[{'batch_size': 16, 'pruning_ratio': 0.40736660487537274, 'quantization': True}, {'batch_size': 8, 'pruning_ratio': 0.4664172302277461, 'quantization': True}, {'batch_size': 16, 'pruning_ratio': 0.40995283896383095, 'quantization': True}, {'batch_size': 8, 'pruning_ratio': 0.3790892340310844, 'quantization': False}, {'batch_size': 8, 'pruning_ratio': 0.3955908947624702, 'quantization': True}, {'batch_size': 16, 'pruning_ratio': 0.2732323673228314, 'quantization': True}, {'batch_size': 16, 'pruning_ratio': 0.42664276918248023, 'quantization': False}, {'batch_size': 8, 'pruning_ratio': 0.4493698995780917, 'quantization': True}, {'batch_size': 16, 'pruning_ratio': 0.29896884480791286, 'quantization': True}, {'batch_size': 24, 'pruning_ratio': 0.48831501368964325, 'quantization': False}, {'batch_size': 24, 'pruning_ratio': 0.4811532961329098, 'quantization': False}, {'batch_size': 24, 'pruning_ratio': 0.48948443913532325, 'quantization': False}, {'batch_size': 24, 'pruning_ratio': 0.489245

In [ ]:
## Task 4: Model deployment on edge device
### Task 4.1: Apply model optimization for each hyperparameter configuration
model_optimization_hyperparameter_list = []
non_model_optimization_hyperparameter_list = []
hyperparameter_judgement_chain = (
    prompt.hyperparameter_judgement_prompt | tool_list.model
)

for filtered_hyperparameter in filtered_hyperparameter_list:
    current_judgement_result = hyperparameter_judgement_chain.invoke(
        filtered_hyperparameter
    )
    current_judgement_result = current_judgement_result.content.lower()

    if current_judgement_result == "yes":
        model_optimization_hyperparameter_list.append(filtered_hyperparameter)
    else:
        non_model_optimization_hyperparameter_list.append(filtered_hyperparameter)

print("The hyperparameters that can be optimized for model optimization are:")
print(model_optimization_hyperparameter_list)
print("The hyperparameters that cannot be optimized for model optimization are:")
print(non_model_optimization_hyperparameter_list)

The hyperparameters that can be optimized for model optimization are:
['pruning_ratio', 'quantization']
The hyperparameters that cannot be optimized for model optimization are:
['batch_size']


In [64]:
### Task 4.2: Apply model optimization for each hyperparameter configuration
quantization_index_list, error_list = [], []

for index in range(len(hyperparameter_configuration_list)):
    hyperparameter_configuration = hyperparameter_configuration_list[index]
    print(hyperparameter_configuration)
    current_agent_configuration = {
        "configurable": {"thread_id": "model_optimization_{}".format(index)}
    }

    # rewrite the code
    current_model_deployment_code_rewrite_prompt = copy.deepcopy(
        prompt.model_deployment_code_rewrite_prompt
    )
    current_model_deployment_code_rewrite_prompt["messages"][0]["content"] = (
        current_model_deployment_code_rewrite_prompt["messages"][0]["content"].format(
            base_deployment_code, local_optimization_code
        )
    )
    current_model_deployment_code_rewrite_prompt["messages"][1]["content"] = (
        current_model_deployment_code_rewrite_prompt["messages"][1]["content"].format(
            index
        )
    )

    agent_output = agent.local_optimization_agent.invoke(
        input=current_model_deployment_code_rewrite_prompt,
        config=current_agent_configuration,
    )
    agent_output = agent_output["messages"][-1].content
    print(agent_output)

    for hyperparameter_name in model_optimization_hyperparameter_list:
        if (
            hyperparameter_name == "quantization"
            and hyperparameter_configuration[hyperparameter_name]
        ):
            quantization_index_list.append(index)
            continue
        hyperparameter_value = hyperparameter_configuration[hyperparameter_name]

        current_model_optimization_addition_prompt = copy.deepcopy(
            prompt.model_optimization_addition_prompt
        )
        current_model_optimization_addition_prompt["messages"][0][
            "content"
        ] = current_model_optimization_addition_prompt["messages"][0]["content"].format(
            hyperparameter_name, hyperparameter_value, hyperparameter_name
        )
        agent_output = agent.local_optimization_agent.invoke(
            input=current_model_optimization_addition_prompt,
            config=current_agent_configuration,
        )
        agent_output = agent_output["messages"][-1].content
        print(agent_output)

    # execute the code and save the model
    try:
        model_optimization_code = tool_list.sanitize_code(agent_output)
        with open("model_optimization.py", "w", encoding="utf-8") as file:
            file.write(model_optimization_code)
        execution_command = "python model_optimization.py"
        status, program_output = code_execution.run(execution_command, True)
    except:
        error_list.append(index)

{'batch_size': 16, 'pruning_ratio': 0.40736660487537274, 'quantization': True}
```python
from transformers import ResNetForImageClassification
import torch

# Load the model
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-152")
model.eval()

# Save the entire model to a local file
torch.save(model, 'model_0.pth')
```
```python
from transformers import ResNetForImageClassification
import torch
import torch_pruning as tp

# Load the model
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-152")
model.eval()

# Example input for the model
example_inputs = torch.randn(1, 3, 224, 224)

# Define the pruning_ratio
pruning_ratio = 0.40736660487537274

# Set up the importance criterion for pruning
imp = tp.importance.GroupNormImportance(p=2)

# Identify layers to ignore during pruning
ignored_layers = []
for m in model.modules():
    if isinstance(m, torch.nn.Linear) and m.out_features == 1000:
        ignored_layers.append(m)

# Initialize the pruner


In [ ]:
## Task 5: Transform the model to the target platform
error_list = []
for index in range(len(hyperparameter_configuration_list)):
    current_agent_configuration = {
        "configurable": {"thread_id": "model_transformation_{}".format(index)}
    }
    try:
        ### Task 5.1: first write some code to transform the PyTorch model into ONNX representation
        current_model_transformation_prompt = copy.deepcopy(
            prompt.model_transformation_prompt
        )
        current_model_transformation_prompt["messages"][0]["content"] = (
            current_model_transformation_prompt["messages"][0]["content"].format(
                dataset_information
            )
        )
        current_model_transformation_prompt["messages"][1]["content"] = (
            current_model_transformation_prompt["messages"][1]["content"].format(
                index, index, deployment_device
            )
        )

        agent_output = agent.local_optimization_agent.invoke(
            input=current_model_transformation_prompt,
            config=current_agent_configuration,
        )

        agent_output = agent_output["messages"][-1].content
        print(agent_output)

        ### Task 5.2: Apply non-model optimization method
        ### generate non-model optimization information
        non_model_optimization_information = ""
        for hyperparameter_name in non_model_optimization_hyperparameter_list:
            non_model_optimization_information = "{}{}* {}: {}".format(
                non_model_optimization_information,
                "" if non_model_optimization_information == "" else "\n",
                hyperparameter_name,
                hyperparameter_configuration_list[index][hyperparameter_name],
            )

        current_non_model_optimization_prompt = copy.deepcopy(
            prompt.non_model_optimization_prompt
        )
        current_non_model_optimization_prompt["messages"][0]["content"] = (
            current_non_model_optimization_prompt["messages"][0]["content"].format(
                non_model_optimization_information
            )
        )

        agent_output = agent.local_optimization_agent.invoke(
            input=current_non_model_optimization_prompt,
            config=current_agent_configuration,
        )

        agent_output = agent_output["messages"][-1].content
        model_transformation_code = tool_list.sanitize_code(agent_output)
        print(model_transformation_code)

        with open("model_transformation.py", "w", encoding="utf-8") as file:
            file.write(model_transformation_code)
        execution_command = "python model_transformation.py"
        status, program_output = code_execution.run(execution_command, True)

        ### Task 5.3: Apply quantization to the model using ONNX
        if index in quantization_index_list:
            from onnxruntime.quantization import quantize_dynamic, QuantType

            quantize_dynamic(
                model_input="model_{}.onnx".format(index),
                model_output="model_{}.onnx".format(index),
                weight_type=QuantType.QUInt8,
            )
    except:
        error_list.append((index, model_transformation_code))

```python
import torch
import torch.onnx
import torchvision.models as models
from torch import nn

# Load the model
model = torch.load('model_0.pth', map_location=torch.device('cpu'))

# Assume that the model is a torchvision model or similar
# Set the model to evaluation mode
model.eval()

# Create a dummy input tensor with the same size as the input images
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX
torch.onnx.export(
    model, 
    dummy_input, 
    "model_0.onnx", 
    export_params=True, 
    opset_version=11, 
    do_constant_folding=True,
    input_names=['input'], 
    output_names=['output']
)
```
import torch
import torch.onnx
import torchvision.models as models
from torch import nn

# Load the model
model = torch.load('model_0.pth', map_location=torch.device('cpu'))

# Assume that the model is a torchvision model or similar
# Set the model to evaluation mode
model.eval()

# Define the batch size
batch_size = 16

# Create a dummy input

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model = torch.load('model_1.pth')
model.eval()

# Dummy input for the model
dummy_input = torch.randn(1, 3, 224, 224)

# Export the model to an ONNX file
torch.onnx.export(model, 
                  dummy_input, 
                  "model_1.onnx", 
                  export_params=True, 
                  opset_version=11, 
                  do_constant_folding=True, 
                  input_names = ['input'], 
                  output_names = ['output'], 
                  dynamic_axes={'input' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}})

print("Model has been successfully exported to model_1.onnx")
```
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model = torch.load('model_1.pth')
model.eval()

# Define the batch size
batch_size = 8

# Dummy input for the model
dummy_input = torch.randn(batch_size, 3, 224, 224

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model_path = 'model_2.pth'
model = torch.load(model_path, map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Define a dummy input matching the input size of the model
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX format
onnx_path = 'model_2.onnx'
torch.onnx.export(
    model, 
    dummy_input, 
    onnx_path, 
    export_params=True, 
    opset_version=11, 
    do_constant_folding=True, 
    input_names=['input'], 
    output_names=['output']
)
```
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model_path = 'model_2.pth'
model = torch.load(model_path, map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Define a dummy input matching the input size of the model with batch_size=16
batch_size = 16
dummy_input = torch.r

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model_path = 'model_3.pth'
model = torch.load(model_path, map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Dummy input for the ONNX export
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX format
onnx_path = 'model_3.onnx'
torch.onnx.export(model, dummy_input, onnx_path, export_params=True, opset_version=11, do_constant_folding=True, input_names=['input'], output_names=['output'])
```
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model_path = 'model_3.pth'
model = torch.load(model_path, map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Hyperparameters
batch_size = 8

# Dummy input for the ONNX export with the specified batch size
dummy_input = torch.randn(batch_size, 3, 224, 224, device='cpu')

# Export the model 

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model = torch.load('model_5.pth', map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Create a dummy input tensor with the same dimensions as the dataset samples
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX format
torch.onnx.export(model, dummy_input, 'model_5.onnx', export_params=True, opset_version=11, do_constant_folding=True, input_names = ['input'], output_names = ['output'])
```
import torch
import torch.onnx
import torchvision.models as models

# Load the model from a local file
model = torch.load('model_5.pth', map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Define the batch size as a hyperparameter
batch_size = 16

# Create a dummy input tensor with the same dimensions as the dataset samples but with the specified batch size
dummy_input = torch.randn(batch_

...

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the PyTorch model
model = torch.load('model_80.pth', map_location=torch.device('cpu'))

# Set the model to inference mode
model.eval()

# Dummy input for the model (based on input size and type)
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX format
torch.onnx.export(model, dummy_input, 'model_80.onnx', export_params=True, opset_version=10, do_constant_folding=True, input_names = ['input'], output_names = ['output'])
```
import torch
import torch.onnx
import torchvision.models as models

# Load the PyTorch model
model = torch.load('model_80.pth', map_location=torch.device('cpu'))

# Set the model to inference mode
model.eval()

# Define the hyperparameters
batch_size = 24

# Dummy input for the model (based on input size, batch size, and type)
dummy_input = torch.randn(batch_size, 3, 224, 224, device='cpu')

# Export the model to ONNX format
torch.onnx.export(model, 

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the model from the local file
model = torch.load('model_82.pth', map_location=torch.device('cpu'))

# Ensure the model is in evaluation mode
model.eval()

# Dummy input for the ONNX export (match the input size of the model)
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX format
torch.onnx.export(
    model,                       # model being run
    dummy_input,                 # model input (or a tuple for multiple inputs)
    'model_82.onnx',             # where to save the model
    export_params=True,          # store the trained parameter weights inside the model file
    opset_version=11,            # the ONNX version to export the model to
    do_constant_folding=True,    # whether to execute constant folding for optimization
    input_names=['input'],       # the model's input names
    output_names=['output'],     # the model's output names
    dynamic_axe

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the trained model from a local file
model = torch.load('model_85.pth', map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Create a dummy input tensor with the same dimensions as the input data
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX format
torch.onnx.export(model, dummy_input, 'model_85.onnx', 
                  export_params=True,  # store the trained parameter weights inside the model file
                  opset_version=11,    # the ONNX version to export the model to
                  do_constant_folding=True,  # whether to execute constant folding for optimization
                  input_names=['input'],  # the model's input names
                  output_names=['output'],  # the model's output names
                  dynamic_axes={'input': {0: 'batch_size'},  # variable length axes
                                'outp

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the trained model from file
model = torch.load('model_88.pth', map_location=torch.device('cpu'))
model.eval()

# Create a dummy input for the ONNX export
dummy_input = torch.randn(1, 3, 224, 224)

# Export the model to an ONNX file
torch.onnx.export(model, dummy_input, 'model_88.onnx', export_params=True, opset_version=11, 
                  do_constant_folding=True, input_names = ['input'], output_names = ['output'])
```
import torch
import torch.onnx
import torchvision.models as models

# Load the trained model from file
model = torch.load('model_88.pth', map_location=torch.device('cpu'))
model.eval()

# Create a dummy input for the ONNX export with batch_size = 8
batch_size = 8
dummy_input = torch.randn(batch_size, 3, 224, 224)

# Export the model to an ONNX file
torch.onnx.export(model, dummy_input, 'model_88.onnx', export_params=True, opset_version=11, 
                  do_constant_folding=True, 

```python
import torch
import torch.onnx
import torchvision.models as models

# Load the model
model_path = 'model_95.pth'
model = torch.load(model_path, map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Dummy input for the model based on the dataset information
dummy_input = torch.randn(1, 3, 224, 224, device='cpu')

# Export the model to ONNX
onnx_path = 'model_95.onnx'
torch.onnx.export(model, dummy_input, onnx_path, export_params=True, opset_version=11, do_constant_folding=True, input_names=['input'], output_names=['output'])
```
import torch
import torch.onnx
import torchvision.models as models

# Load the model
model_path = 'model_95.pth'
model = torch.load(model_path, map_location=torch.device('cpu'))

# Set the model to evaluation mode
model.eval()

# Hyperparameters
batch_size = 8

# Dummy input for the model based on the dataset information and batch size
dummy_input = torch.randn(batch_size, 3, 224, 224, device='cpu')

# Export the model t

In [67]:
## Task 6: Generate code for performance monitoring
### Task 6.1: retrieve reference code
performance_reference_code = get_code_reference(
    mode=False, user_requirements=performance_requirement
)

print("Retrieved reference code for resource monitoring:")
print("\033[34m{}\033[0m".format(performance_reference_code))

Retrieved reference code for resource monitoring:
1. Reference code for inference latency monitoring
```python
import time
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_time_list = []

# Suppose the test_loader contains all the test data
with torch.no_grad():
    for test_data, test_label in test_loader:
        test_data, test_label = test_data.to(device), test_label.to(device)
        batch_size = test_data.size(0)

        # Perform inference on the test data
        start_time = time.time()
        output = model(test_data)
        end_time = time.time()

# Inference latency per sample
average_inference_time = sum(inference_time_list) / len(inference_time_list) / batch_size * 1000
print(f"Average inference latency per sample: {average_inference_time} ms")
```

2. Reference code for throughput monitoring
```python
import time
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_time_list = []

#

In [9]:
resource_reference_code_list = resource_reference_code.split("```\n")
for index in range(len(resource_reference_code_list) - 1):
    resource_reference_code_list[index] = resource_reference_code_list[index] + "```"

performance_reference_code_list = performance_reference_code.split("```\n")
for index in range(len(performance_reference_code_list) - 1):
    performance_reference_code_list[index] = (
        performance_reference_code_list[index] + "```"
    )

In [41]:
### Task 6.2: generate code for local deployment and monitoring
edge_device_deployment_code_list = []
current_agent_configuration = {"configurable": {"thread_id": "edge_deployment"}}

#### Task 6.2.1: generate code for onnx model loading
current_ONNX_model_loading_prompt = copy.deepcopy(prompt.ONNX_model_loading_prompt)
current_ONNX_model_loading_prompt["messages"][1]["content"] = (
    current_ONNX_model_loading_prompt["messages"][1]["content"]
)
agent_output = agent.edge_model_deployment_agent.invoke(
    input=current_ONNX_model_loading_prompt,
    config=current_agent_configuration,
)
agent_output = agent_output["messages"][-1].content
print(agent_output)

```python
import onnx
import onnxruntime as ort

# Load the ONNX model
model_path = 'model_file.onnx'
onnx_model = onnx.load(model_path)

# Verify the model's structure and confirm that it has a valid schema
onnx.checker.check_model(onnx_model)

# Initialize an ONNX Runtime session
ort_session = ort.InferenceSession(model_path)

print("ONNX model loaded and checked successfully.")
```


In [43]:
#### Task 6.2.2: generate code for model inference on local dataset
current_edge_model_inference_prompt = copy.deepcopy(prompt.edge_model_inference_prompt)
current_edge_model_inference_prompt["messages"][0][
    "content"
] = current_edge_model_inference_prompt["messages"][0]["content"].format(
    dataset_information, local_dataset_path, non_model_optimization_hyperparameter_list
)
agent_output = agent.edge_model_deployment_agent.invoke(
    input=current_edge_model_inference_prompt,
    config=current_agent_configuration,
)
agent_output = agent_output["messages"][-1].content
print(agent_output)

```python
import onnx
import onnxruntime as ort
import numpy as np

# Load the ONNX model
model_path = 'model_file.onnx'
onnx_model = onnx.load(model_path)

# Verify the model's structure and confirm that it has a valid schema
onnx.checker.check_model(onnx_model)

# Initialize an ONNX Runtime session
ort_session = ort.InferenceSession(model_path)

# Load the testing dataset
data_path = './dataset/ILSVRC'
test_data = np.load(f'{data_path}/test_data.npy')
test_labels = np.load(f'{data_path}/test_label.npy')

# Hyperparameters
batch_size = 32

# Perform inference in batches
num_batches = len(test_data) // batch_size
all_predictions = []

for i in range(num_batches):
    batch_data = test_data[i * batch_size:(i + 1) * batch_size]
    ort_inputs = {ort_session.get_inputs()[0].name: batch_data}
    batch_predictions = ort_session.run(None, ort_inputs)
    all_predictions.append(batch_predictions)

print("Inference completed on the test dataset.")
```


In [44]:
#### Task 6.2.3: add two arguments: model weight path and dataset path
agent_output = agent.edge_model_deployment_agent.invoke(
    input=prompt.edge_model_argument_addition_prompt,
    config=current_agent_configuration,
)
agent_output = agent_output["messages"][-1].content
print(agent_output)

```python
import onnx
import onnxruntime as ort
import numpy as np
import argparse

# Parse command-line arguments
parser = argparse.ArgumentParser(description='ONNX Model Inference')
parser.add_argument('--model_weight_path', type=str, required=True, help='Path to the ONNX model weight file')
parser.add_argument('--dataset_path', type=str, required=True, help='Path to the dataset directory')
args = parser.parse_args()

# Load the ONNX model
onnx_model = onnx.load(args.model_weight_path)

# Verify the model's structure and confirm that it has a valid schema
onnx.checker.check_model(onnx_model)

# Initialize an ONNX Runtime session
ort_session = ort.InferenceSession(args.model_weight_path)

# Load the testing dataset
test_data = np.load(f'{args.dataset_path}/test_data.npy')
test_labels = np.load(f'{args.dataset_path}/test_label.npy')

# Hyperparameters
batch_size = 32

# Perform inference in batches
num_batches = len(test_data) // batch_size
all_predictions = []

for i in range(num_batc

In [47]:
current_non_model_optimization_argument_prompt = copy.deepcopy(
    prompt.non_model_optimization_argument_prompt
)
current_non_model_optimization_argument_prompt["messages"][0]["content"] = (
    current_non_model_optimization_argument_prompt["messages"][0]["content"].format(
        non_model_optimization_hyperparameter_list
    )
)
agent_output = agent.edge_model_deployment_agent.invoke(
    input=current_non_model_optimization_argument_prompt,
    config=current_agent_configuration,
)
agent_output = agent_output["messages"][-1].content
print(agent_output)

```python
import onnx
import onnxruntime as ort
import numpy as np
import argparse

# Parse command-line arguments
parser = argparse.ArgumentParser(description='ONNX Model Inference')
parser.add_argument('--model_weight_path', type=str, required=True, help='Path to the ONNX model weight file')
parser.add_argument('--dataset_path', type=str, required=True, help='Path to the dataset directory')
parser.add_argument('--batch_size', type=int, default=32, help='Batch size for inference')
args = parser.parse_args()

# Load the ONNX model
onnx_model = onnx.load(args.model_weight_path)

# Verify the model's structure and confirm that it has a valid schema
onnx.checker.check_model(onnx_model)

# Initialize an ONNX Runtime session
ort_session = ort.InferenceSession(args.model_weight_path)

# Load the testing dataset
test_data = np.load(f'{args.dataset_path}/test_data.npy')
test_labels = np.load(f'{args.dataset_path}/test_label.npy')

# Hyperparameters
batch_size = args.batch_size

# Perform infer

In [48]:
#### Task 6.2.5: add resource monitoring code
for single_resource_reference_code in resource_reference_code_list:
    current_edge_resource_monitoring_prompt = copy.deepcopy(
        prompt.edge_resource_monitoring_prompt
    )
    current_edge_resource_monitoring_prompt["messages"][0]["content"] = (
        current_edge_resource_monitoring_prompt["messages"][0]["content"].format(
            resource_reference_code
        )
    )
    agent_output = agent.edge_model_deployment_agent.invoke(
        input=current_edge_resource_monitoring_prompt,
        config=current_agent_configuration,
    )
    agent_output = agent_output["messages"][-1].content
    print(agent_output)

```python
import onnx
import onnxruntime as ort
import numpy as np
import argparse
import os
import psutil
import threading
import multiprocessing
import time

# Function to monitor resource consumption
def monitor_resources(finished, cpu_memory_list):
    process = psutil.Process(os.getpid())
    while not finished.is_set():
        time.sleep(1)
        cpu_memory = process.memory_info().rss / 1024 / 1024  # in MB
        cpu_memory_list.append(cpu_memory)

# Function for model inference
def model_inference(args):
    # Load the ONNX model
    onnx_model = onnx.load(args.model_weight_path)
    onnx.checker.check_model(onnx_model)
    ort_session = ort.InferenceSession(args.model_weight_path)

    # Load the testing dataset
    test_data = np.load(f'{args.dataset_path}/test_data.npy')
    test_labels = np.load(f'{args.dataset_path}/test_label.npy')

    # Hyperparameters
    batch_size = args.batch_size

    # Resource monitoring setup
    finished = threading.Event()
    cpu_memory_l

In [ ]:
#### Task 6.2.6: add performance monitoring code
for single_performance_monitoring_code in performance_reference_code_list:
    current_edge_performance_monitoring_prompt = copy.deepcopy(
        prompt.edge_performance_monitoring_prompt
    )
    current_edge_performance_monitoring_prompt["messages"][0]["content"] = (
        current_edge_performance_monitoring_prompt["messages"][0]["content"].format(
            single_performance_monitoring_code
        )
    )
    agent_output = agent.edge_model_deployment_agent.invoke(
        input=current_edge_performance_monitoring_prompt,
        config=current_agent_configuration,
    )
    agent_output = agent_output["messages"][-1].content
    print(agent_output)

edge_device_deployment_code = tool_list.sanitize_code(agent_output)
with open("edge_deployment.py", "w", encoding="utf-8") as file:
    file.write(edge_device_deployment_code)

```python
import onnx
import onnxruntime as ort
import numpy as np
import argparse
import os
import psutil
import threading
import multiprocessing
import time

# Function to monitor resource consumption
def monitor_resources(finished_flag, cpu_memory_list):
    process = psutil.Process(os.getpid())
    while not finished_flag.is_set():
        time.sleep(1)
        cpu_memory = process.memory_info().rss / 1024 / 1024  # in MB
        cpu_memory_list.append(cpu_memory)

# Function for model inference
def model_inference(args, finished_flag, cpu_memory_list):
    # Load the ONNX model
    onnx_model = onnx.load(args.model_weight_path)
    onnx.checker.check_model(onnx_model)
    ort_session = ort.InferenceSession(args.model_weight_path)

    # Load the testing dataset
    test_data = np.load(f'{args.dataset_path}/test_data.npy')
    test_labels = np.load(f'{args.dataset_path}/test_label.npy')

    # Hyperparameters
    batch_size = args.batch_size

    # Start the monitoring thread
    m

In [9]:
# =================== MODULE 2 ===================
## Task 7: Cross platform deployment and monitoring
validated_hyperparameter_configuration_list = []
command_template = "python3 {} --model_weight_path {} --dataset_path {} {} > log.txt"
error_list, filtered_program_output_list = [], []

for index in range(len(hyperparameter_configuration_list)):
    try:
        ### Task 7.1: Transfer model weights to the target device
        cross_platform_protocol.send_file(
            file_path="E:/models/model_{}.onnx".format(index),
            url="{}:{}/model_transmission".format(config.server_address, config.port),
            meta_information={
                "model_name": model_name,
                "model_weight_name": "model_{}.onnx".format(index),
                "model_size": os.path.getsize("E:/models/model_{}.onnx".format(index)),
            },
        )

        ### Task 7.2: Transfer execution script to the target device
        cross_platform_protocol.send_file(
            file_path="edge_deployment.py",
            url="{}:{}/script_transmission".format(config.server_address, config.port),
            meta_information={"script_name": "edge_deployment_{}.py".format(index)},
        )

        ### Task 7.3: Execute the script on the target device
        current_non_model_optimization_argument = ""
        for hyperparameter_name in non_model_optimization_hyperparameter_list:
            hyperparameter_value = hyperparameter_configuration_list[index][
                hyperparameter_name
            ]
            current_non_model_optimization_argument = "{} --{} {}".format(
                current_non_model_optimization_argument,
                hyperparameter_name,
                hyperparameter_value,
            )
        current_non_model_optimization_argument = (
            current_non_model_optimization_argument.strip()
        )

        execution_result = cross_platform_protocol.send_command(
            url="{}:{}/model_execution".format(config.server_address, config.port),
            meta_information={
                "command": command_template.format(
                    "{}", "{}", "{}", current_non_model_optimization_argument
                ),
                "model_weight_name": "model_{}.onnx".format(index),
                "script_name": "edge_deployment_{}.py".format(index),
            },
        )

        ### Task 7.4: execution result filtering
        edge_model_monitoring_output_filtering_chain = (
            prompt.model_monitoring_output_filtering_prompt | tool_list.model
        )
        filtered_program_output = edge_model_monitoring_output_filtering_chain.invoke(
            execution_result
        )
        filtered_program_output = filtered_program_output.content
        filtered_program_output_list.append(filtered_program_output)
    except Exception as e:
        print(e)
        error_list.append(index)

<Response [200]>
Status: True | Message: Model weight received successfully.
<Response [200]>
Status: True | Message: Script received successfully.
<Response [200]>
Status: True | Message: Model execution completed. | Result: Maximum GPU memory usage: 0.0 MB
Model size in local storage: 34.396371841430664 MB
Average inference latency per sample: 277.333961725235 ms
Throughput per second: 3.197256014642237 samples per second
Classification accuracy: 0.001
Precision: 1.2213964225298786e-06
Recall rate: 0.0011389521640091116
F1-score: 2.4401760342991145e-06

<Response [200]>
Status: True | Message: Model weight received successfully.
<Response [200]>
Status: True | Message: Script received successfully.
<Response [200]>
Status: True | Message: Model execution completed. | Result: Maximum GPU memory usage: 0.0 MB
Model size in local storage: 29.191871643066406 MB
Average inference latency per sample: 234.67691266536713 ms
Throughput per second: 3.9957642786018153 samples per second
Classif

In [ ]:
### Task 7.5: judge whether the model performance meets the requirements
for index in range(len(filtered_program_output_list)):
    filtered_program_output = filtered_program_output_list[index]

    performance_requirement_judgement_chain = (
        prompt.performance_requirement_judgement_prompt | tool_list.model
    )

    judgement_result = performance_requirement_judgement_chain.invoke(
        input={
            "performance_requirement": performance_requirement,
            "performance_monitoring_result": filtered_program_output,
        }
    )

    judgement_result = judgement_result.content

    if judgement_result.lower() == "yes":

        print("The {}-th model meets the performance requirements.".format(index))

        validated_hyperparameter_configuration_list.append(
            (index, filtered_program_output)
        )

In [ ]:
## Task 8: print all the hyperparameter configurations that meet the requirements
print(
    "After several iterations of self-optimization and deployment, the following hyperparameter configurations meet all of your requirements:"
)
for index, performance_monitoring_result in validated_hyperparameter_configuration_list:
    print("The {}-th hyperparameter configuration:".format(index))
    print(hyperparameter_configuration_list[index])
    print("The performance monitoring result:")
    print(performance_monitoring_result)
    print("\n================================================\n")

print("Please choose the best hyperparameter configuration for the final deployment.")
print("Please enter the index of the best hyperparameter configuration:")
best_hyperparameter_index = int(input())

In [ ]:
# =================== MODULE 3 ===================
# Module 3: Model Deployment and Monitoring on Edge Device
# Based on the AutoDP paper, we implement three technical modules:
#   1. Agent-Based Operator Analyzer (§3.1)
#   2. Structure-Aware Pattern Detector (§3.2)
#   3. Greedy Cross-Engine Inference Algorithm (§3.3)

import onnx
import numpy as np
from langchain_chroma import Chroma

## Task 9: Retrieve the best optimized model and prepare for AutoDP deployment
best_hyperparameter_configuration = hyperparameter_configuration_list[
    best_hyperparameter_index
]
optimized_model_path = "model_{}.onnx".format(best_hyperparameter_index)

# Load the ONNX model and extract its computation graph
onnx_model = onnx.load(optimized_model_path)
model_graph = onnx_model.graph

# Extract all operators (nodes) from the model computation graph
model_operators = []
for node in model_graph.node:
    model_operators.append({
        "name": node.name,
        "op_type": node.op_type,
        "inputs": list(node.input),
        "outputs": list(node.output),
        "attributes": {attr.name: str(attr) for attr in node.attribute},
    })

print("Loaded ONNX model from: {}".format(optimized_model_path))
print("Total number of operators in the model: {}".format(len(model_operators)))
print("Operator types: {}".format(set(op["op_type"] for op in model_operators)))

In [ ]:
# =================== Technical Module 1: Agent-Based Operator Analyzer (§3.1) ===================
# This module checks the compatibility between each operator in the model
# and the inference engines available on the target embedded device.
# Two stages: (1) Information Retrieval with Maintenance, (2) Compatibility Check via Code Analysis.

# Step 1: Define the list of available on-device inference engines
# Each engine entry contains its name, GitHub repository URL, and documentation URL.
available_engines = [
    {
        "name": "ONNX Runtime",
        "short_name": "ort",
        "repo_url": "https://github.com/microsoft/onnxruntime",
        "doc_url": "https://onnxruntime.ai/docs/",
    },
    {
        "name": "OpenVINO",
        "short_name": "openvino",
        "repo_url": "https://github.com/openvinotoolkit/openvino",
        "doc_url": "https://docs.openvino.ai/",
    },
    {
        "name": "TensorRT",
        "short_name": "tensorrt",
        "repo_url": "https://github.com/NVIDIA/TensorRT",
        "doc_url": "https://docs.nvidia.com/deeplearning/tensorrt/",
    },
]

# Initialize the operator knowledge database (vector store) for storing
# engine-supported operator attributes: name, functionality, I/O specs, and code.
operator_knowledge_db = Chroma(
    collection_name="autodp_operator_knowledge",
    embedding_function=tool_list.embeddings,
    persist_directory="./chroma_langchain_db",
)

print("Initialized AutoDP with {} available inference engines.".format(len(available_engines)))
for engine in available_engines:
    print("  - {} ({})".format(engine["name"], engine["repo_url"]))

In [ ]:
# Stage 1: Information Retrieval with Maintenance (§3.1)
# For each on-device engine, the agent retrieves its source code and documentation
# to identify supported operators and extract their implementation details.
# The extracted information is stored in the operator knowledge database.

# Prompt P.1: Operator Implementation Code Extraction
operator_code_extraction_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a code analyzer specialized in understanding inference engines "
                "(e.g., ONNX Runtime, OpenVINO, TensorRT). Your task is to locate and extract "
                "the implementation code segment corresponding to a given operator.\n\n"
                "Objectives:\n"
                "1. Search the repository and locate the exact file(s) and function(s) that "
                "implement the given operator.\n"
                "2. Extract the relevant code segment that includes:\n"
                "   - The operator class or kernel definition.\n"
                "   - Its core compute function or forward implementation.\n"
                "   - Input/output data types, dimensions, and semantics of the operator.\n"
                "   - Key attributes or registration macros that link to the operator name.\n"
                "3. Summarize and output the extracted code segment in a structured format:\n"
                "   - Operator Name: <name>\n"
                "   - Functionality: <brief description of what this operator computes>\n"
                "   - I/O Specifications: <input/output tensor types, shapes, and semantics>\n"
                "   - Implementation Code: ```<code>```\n\n"
                "Inference Engine: {engine_name}\n"
                "Repository URL: {repo_url}\n"
                "Operator Name: {operator_name}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Please search for the implementation of the '{operator_name}' operator in the "
                "{engine_name} inference engine. Extract and return its implementation details "
                "following the structured format specified above."
            ),
        },
    ]
}

# Prompt for retrieving the list of supported operators from an engine
engine_operator_list_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a knowledgeable AI assistant specialized in inference engines for "
                "embedded AI deployment. Your task is to retrieve the complete list of "
                "supported operators for a given inference engine.\n\n"
                "Inference Engine: {engine_name}\n"
                "Documentation URL: {doc_url}\n"
                "Repository URL: {repo_url}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Please search for and retrieve the complete list of operators supported by "
                "{engine_name}. For each operator, provide:\n"
                "1. Operator name\n"
                "2. Brief functionality description\n\n"
                "Output the result as a JSON list of objects with keys 'name' and 'functionality'. "
                "Only output the JSON list without extra contents."
            ),
        },
    ]
}

print("Defined prompts for operator information retrieval (P.1).")

In [ ]:
# Stage 1 (cont.): Retrieve supported operators for each engine and store in database
# For each engine, the agent searches for supported operators and extracts their
# implementation details: operator name, functionality, I/O specs, and code (Eq. 1-2).

from langchain_core.documents import Document

# Create the operator retrieval agent (ReAct agent with web search tool)
operator_retrieval_agent = create_react_agent(
    tool_list.model,
    [tool_list.web_search_tool],
    checkpointer=tool_list.agent_memory,
)

# D = {O_e1, O_e2, ...} - the operator knowledge database
engine_operator_registry = {}  # engine_name -> list of operator dicts

for engine in available_engines:
    engine_name = engine["name"]
    print("\n=== Retrieving supported operators for {} ===".format(engine_name))

    # Step 1a: Retrieve the list of supported operators for this engine
    current_operator_list_prompt = copy.deepcopy(engine_operator_list_prompt)
    current_operator_list_prompt["messages"][0]["content"] = (
        current_operator_list_prompt["messages"][0]["content"]
        .replace("{engine_name}", engine_name)
        .replace("{doc_url}", engine["doc_url"])
        .replace("{repo_url}", engine["repo_url"])
    )
    current_operator_list_prompt["messages"][1]["content"] = (
        current_operator_list_prompt["messages"][1]["content"]
        .replace("{engine_name}", engine_name)
    )

    operator_list_agent_config = {
        "configurable": {"thread_id": "operator_list_{}".format(engine["short_name"])}
    }
    operator_list_response = operator_retrieval_agent.invoke(
        input={"messages": current_operator_list_prompt["messages"]},
        config=operator_list_agent_config,
    )
    operator_list_text = operator_list_response["messages"][-1].content

    # Parse the JSON list of operators from the agent's response
    try:
        # Extract JSON from the response (handle markdown code blocks)
        json_start = operator_list_text.find("[")
        json_end = operator_list_text.rfind("]") + 1
        if json_start != -1 and json_end > json_start:
            engine_operators = json.loads(operator_list_text[json_start:json_end])
        else:
            engine_operators = json.loads(operator_list_text)
    except json.JSONDecodeError:
        print("Warning: Failed to parse operator list for {}. Using LLM to retry...".format(engine_name))
        # Reflexion: retry with a more explicit prompt
        retry_response = operator_retrieval_agent.invoke(
            input={"messages": [
                {"role": "user", "content": (
                    "The previous response could not be parsed as JSON. Please output ONLY a valid "
                    "JSON list of supported operators for {}. Format: "
                    '[{{"name": "OpName", "functionality": "description"}}, ...]. '
                    "No extra text."
                ).format(engine_name)}
            ]},
            config=operator_list_agent_config,
        )
        retry_text = retry_response["messages"][-1].content
        json_start = retry_text.find("[")
        json_end = retry_text.rfind("]") + 1
        engine_operators = json.loads(retry_text[json_start:json_end])

    print("Retrieved {} supported operators for {}.".format(len(engine_operators), engine_name))

    # Step 1b: For each operator, extract detailed implementation information
    # O_ej = {o_j_1, o_j_2, ...}, o_j_i = <name, functionality, IO, code>
    engine_operator_details = []
    for op_info in engine_operators:
        op_name = op_info["name"]
        op_functionality = op_info.get("functionality", "")

        # Use the agent to extract implementation details (P.1)
        current_extraction_prompt = copy.deepcopy(operator_code_extraction_prompt)
        current_extraction_prompt["messages"][0]["content"] = (
            current_extraction_prompt["messages"][0]["content"]
            .replace("{engine_name}", engine_name)
            .replace("{repo_url}", engine["repo_url"])
            .replace("{operator_name}", op_name)
        )
        current_extraction_prompt["messages"][1]["content"] = (
            current_extraction_prompt["messages"][1]["content"]
            .replace("{operator_name}", op_name)
            .replace("{engine_name}", engine_name)
        )

        extraction_agent_config = {
            "configurable": {"thread_id": "op_extract_{}_{}".format(engine["short_name"], op_name)}
        }
        extraction_response = operator_retrieval_agent.invoke(
            input={"messages": current_extraction_prompt["messages"]},
            config=extraction_agent_config,
        )
        extraction_text = extraction_response["messages"][-1].content

        # Structure the operator attributes
        operator_entry = {
            "name": op_name,
            "functionality": op_functionality,
            "io_specs": extraction_text,  # Contains I/O specs from the extracted info
            "code": extraction_text,       # Full extracted implementation details
            "engine": engine_name,
        }
        engine_operator_details.append(operator_entry)

        # Store in the vector database for retrieval
        doc_content = (
            "Engine: {}\nOperator: {}\nFunctionality: {}\n"
            "Details:\n{}"
        ).format(engine_name, op_name, op_functionality, extraction_text)

        operator_knowledge_db.add_documents(
            documents=[Document(
                page_content=doc_content,
                metadata={
                    "engine": engine_name,
                    "operator": op_name,
                    "type": "operator_implementation",
                },
            )]
        )

    engine_operator_registry[engine_name] = engine_operator_details
    print("Stored {} operator details for {} in the knowledge database.".format(
        len(engine_operator_details), engine_name
    ))

In [ ]:
# Stage 2: Compatibility Check via Code Analysis (§3.1)
# For each operator in the model, check compatibility with each engine's supported operators.
# The agent analyzes functionality equivalence and I/O match (P.2, Eq. 3-4).

# Prompt P.2: Compatibility Check
compatibility_check_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a code auditor specialized in inference engines. Your goal is to "
                "determine whether an operator from a pretrained model is compatible with "
                "an engine-supported operator by analyzing their source code and specifications.\n\n"
                "Objectives:\n"
                "1. Understand operator functionality\n"
                "   * Determine what computational operation they perform.\n"
                "   * Compare their mathematical formulations, core loops, or library calls.\n"
                "   * Confirm if both operators achieve the same computational semantics.\n"
                "2. Analyze I/O specifications\n"
                "   * Extract the list of input and output tensors.\n"
                "   * Record each tensor's dimensionality, data type, and role.\n"
                "   * Identify and note any mismatch.\n"
                "3. If both the computational functionality and I/O specifications match, "
                "only output 'yes'; otherwise, only output 'no'.\n\n"
                "Pretrained model operator:\n{model_operator_info}\n\n"
                "Engine-supported operator:\n{engine_operator_info}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Please analyze the compatibility between the pretrained model operator and "
                "the engine-supported operator described above. Only output 'yes' or 'no'."
            ),
        },
    ]
}

# Build the compatibility matrix C[n x k] (Eq. 3-4)
# C[i, j] = 1 means operator o_i is supported by engine e_j
num_operators = len(model_operators)
num_engines = len(available_engines)
compatibility_matrix = np.zeros((num_operators, num_engines), dtype=int)

# For each model operator, check compatibility with each engine
# M ~ {o_1, o_2, ...}, o_i ~ [C_1_i, C_2_i, ...]
operator_engine_mapping = []  # list of dicts: {op_index, op_type, compatible_engines}

for i, model_op in enumerate(model_operators):
    op_type = model_op["op_type"]
    compatible_engines = []

    # Prepare model operator info string
    model_op_info = (
        "Operator Name: {}\nOperator Type: {}\n"
        "Inputs: {}\nOutputs: {}\nAttributes: {}"
    ).format(
        model_op["name"], model_op["op_type"],
        model_op["inputs"], model_op["outputs"],
        json.dumps(model_op["attributes"], indent=2),
    )

    for j, engine in enumerate(available_engines):
        engine_name = engine["name"]
        engine_ops = engine_operator_registry.get(engine_name, [])

        # First, try exact name matching for efficiency
        exact_match = False
        for engine_op in engine_ops:
            if engine_op["name"].lower() == op_type.lower():
                exact_match = True
                compatibility_matrix[i, j] = 1
                compatible_engines.append(engine_name)
                break

        # If no exact match, use the LLM agent to check semantic compatibility (P.2)
        if not exact_match:
            # Retrieve the most relevant engine operators from the database
            relevant_docs = operator_knowledge_db.similarity_search(
                "operator {} in {} engine".format(op_type, engine_name),
                k=3,
                filter={"engine": engine_name},
            )

            if relevant_docs:
                engine_op_info = "\n\n".join(doc.page_content for doc in relevant_docs)

                # Apply the compatibility check prompt (P.2)
                current_compat_prompt = copy.deepcopy(compatibility_check_prompt)
                current_compat_prompt["messages"][0]["content"] = (
                    current_compat_prompt["messages"][0]["content"]
                    .replace("{model_operator_info}", model_op_info)
                    .replace("{engine_operator_info}", engine_op_info)
                )

                compat_response = tool_list.model.invoke(
                    [{"role": msg["role"], "content": msg["content"]}
                     for msg in current_compat_prompt["messages"]]
                )
                compat_result = compat_response.content.strip().lower()

                # Reflexion: if the response is ambiguous, retry once
                if compat_result not in ["yes", "no"]:
                    reflexion_response = tool_list.model.invoke(
                        [{"role": "user", "content": (
                            "Your previous response '{}' was not clear. "
                            "Is the model operator '{}' compatible with {} engine? "
                            "Only output 'yes' or 'no'."
                        ).format(compat_result, op_type, engine_name)}]
                    )
                    compat_result = reflexion_response.content.strip().lower()

                if compat_result == "yes":
                    compatibility_matrix[i, j] = 1
                    compatible_engines.append(engine_name)

    operator_engine_mapping.append({
        "op_index": i,
        "op_type": op_type,
        "compatible_engines": compatible_engines,
    })

    if (i + 1) % 20 == 0 or i == num_operators - 1:
        print("Compatibility check progress: {}/{} operators analyzed.".format(i + 1, num_operators))

# Store the compatibility results in the database for reuse
for i, mapping in enumerate(operator_engine_mapping):
    operator_knowledge_db.add_documents(
        documents=[Document(
            page_content="Model Operator: {} | Compatible Engines: {}".format(
                mapping["op_type"], ", ".join(mapping["compatible_engines"])
            ),
            metadata={
                "operator": mapping["op_type"],
                "type": "compatibility_result",
                "compatible_engines": str(mapping["compatible_engines"]),
            },
        )]
    )

print("\n=== Compatibility Matrix Summary ===")
for j, engine in enumerate(available_engines):
    supported_count = int(np.sum(compatibility_matrix[:, j]))
    print("{}: {}/{} operators supported ({:.1f}%)".format(
        engine["name"], supported_count, num_operators,
        100.0 * supported_count / num_operators,
    ))

# Check if all operators are collectively supported by the union of engines
collectively_supported = np.any(compatibility_matrix, axis=1)
unsupported_ops = [model_operators[i]["op_type"] for i in range(num_operators) if not collectively_supported[i]]
if unsupported_ops:
    print("\nWarning: {} operators are not supported by any engine: {}".format(
        len(unsupported_ops), unsupported_ops
    ))
    print("These operators will require custom registration or CPU fallback.")
else:
    print("\nAll operators are collectively supported by the available engines.")

In [ ]:
# Stage 2 (cont.): Handle developer-customized operators
# For operators not supported by any engine, instruct the agent to generate
# registration code for the target engines (as mentioned in §3.1 Remarks).

custom_operator_registration_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are an expert in inference engine development and custom operator registration. "
                "A pretrained AI model contains an operator that is not natively supported by any "
                "available inference engine. Your task is to generate Python code that registers "
                "this custom operator into a specified inference engine so it can be executed.\n\n"
                "The code should:\n"
                "1. Define the custom operator's compute kernel.\n"
                "2. Register it with the engine's custom operator API.\n"
                "3. Handle input/output tensor shapes and data types correctly.\n"
                "4. Be self-contained and directly executable."
            ),
        },
        {
            "role": "user",
            "content": (
                "Please generate Python code to register the following custom operator into "
                "the {engine_name} inference engine.\n\n"
                "Operator Details:\n{operator_info}\n\n"
                "Please output the registration code directly."
            ),
        },
    ]
}

# Generate custom operator registration code for unsupported operators
custom_operator_code = {}  # op_type -> {engine_name: registration_code}

for i in range(num_operators):
    if not collectively_supported[i]:
        op_type = model_operators[i]["op_type"]
        if op_type in custom_operator_code:
            continue  # Already handled this operator type

        print("Generating custom registration for unsupported operator: {}".format(op_type))
        custom_operator_code[op_type] = {}

        model_op_info = (
            "Operator Name: {}\nOperator Type: {}\n"
            "Inputs: {}\nOutputs: {}\nAttributes: {}"
        ).format(
            model_operators[i]["name"], op_type,
            model_operators[i]["inputs"], model_operators[i]["outputs"],
            json.dumps(model_operators[i]["attributes"], indent=2),
        )

        # Try to register with the most capable engine (ONNX Runtime as fallback)
        target_engine = available_engines[0]  # Default: ONNX Runtime
        current_reg_prompt = copy.deepcopy(custom_operator_registration_prompt)
        current_reg_prompt["messages"][1]["content"] = (
            current_reg_prompt["messages"][1]["content"]
            .replace("{engine_name}", target_engine["name"])
            .replace("{operator_info}", model_op_info)
        )

        reg_agent_config = {
            "configurable": {"thread_id": "custom_reg_{}".format(op_type)}
        }
        reg_response = operator_retrieval_agent.invoke(
            input={"messages": current_reg_prompt["messages"]},
            config=reg_agent_config,
        )
        reg_code = reg_response["messages"][-1].content
        custom_operator_code[op_type][target_engine["name"]] = reg_code

        # Mark this operator as compatible with the target engine after registration
        compatibility_matrix[i, 0] = 1  # ONNX Runtime (index 0)
        operator_engine_mapping[i]["compatible_engines"].append(target_engine["name"])

if custom_operator_code:
    print("\nGenerated custom registration code for {} operator types.".format(
        len(custom_operator_code)
    ))
else:
    print("\nNo custom operator registration needed - all operators are natively supported.")

In [ ]:
# =================== Technical Module 2: Structure-Aware Pattern Detector (§3.2) ===================
# This module identifies optimizable operator patterns (e.g., operator fusion, redundancy removal)
# from each engine's source code and reorganizes model copies accordingly.
# Two stages: (1) Pattern Mining, (2) Model Reorganization.

# Stage 1: Pattern Mining (Eq. 5)
# For each engine, parse documentation to identify internal optimizable patterns
# P_j = {p_j_1, p_j_2, ...}, p_j_r = <O_j_r, c_j_r>

pattern_mining_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are an expert in inference engine internals and graph optimization. "
                "Your task is to identify all internal optimizable operator patterns "
                "(e.g., operator fusion, operator elimination, constant folding) used by "
                "a given inference engine.\n\n"
                "For each pattern, provide:\n"
                "1. Pattern Name: the name of the optimization pattern\n"
                "2. Operator Sequence: the sequence of consecutive operators that form this pattern "
                "(e.g., ['Conv', 'Add'] for Conv+Add fusion)\n"
                "3. Optimization Type: 'fusion' (merge operators) or 'elimination' (remove redundant)\n"
                "4. Description: what the optimization does\n\n"
                "Inference Engine: {engine_name}\n"
                "Repository URL: {repo_url}\n"
                "Documentation URL: {doc_url}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Please search for and identify all optimizable operator patterns in the "
                "{engine_name} inference engine. Output the result as a JSON list of objects "
                "with keys 'pattern_name', 'operator_sequence' (a list of operator type strings), "
                "'optimization_type', and 'description'. Only output the JSON list."
            ),
        },
    ]
}

# Prompt P.3: Code Rewriting for Pattern Application
pattern_code_rewriting_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "Given a code snippet for operator-level optimization (fusion and removal) used "
                "by an engine, you need to rewrite this snippet so that it becomes a standalone "
                "Python function that can be directly applied to an ONNX model graph.\n\n"
                "Rules:\n"
                "1. Abstract away engine-specific dependencies\n"
                "   * Remove references to build configs or environment-specific libraries.\n"
                "   * Replace engine API calls with generic ONNX graph operations.\n"
                "2. Preserve optimization semantics\n"
                "   * Maintain the same operator transformation (e.g., Conv+Add -> Conv with bias).\n"
                "3. Code should be organized in a structured function\n"
                "   * Input: an ONNX ModelProto object.\n"
                "   * Output: a modified ONNX ModelProto with the optimization applied.\n"
                "4. Use the 'onnx' and 'onnx.helper' libraries for graph manipulation.\n\n"
                "Raw optimization pattern: {pattern_info}\n"
                "Engine: {engine_name}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Please write a standalone Python function named 'apply_{pattern_name}' that "
                "takes an ONNX ModelProto as input and applies the '{pattern_name}' optimization "
                "pattern. The function should detect the operator sequence in the graph and apply "
                "the transformation. Output only the Python code."
            ),
        },
    ]
}

# Mine optimizable patterns for each engine
engine_optimizable_patterns = {}  # engine_name -> list of pattern dicts

for engine in available_engines:
    engine_name = engine["name"]
    print("\n=== Mining optimizable patterns for {} ===".format(engine_name))

    current_pattern_prompt = copy.deepcopy(pattern_mining_prompt)
    current_pattern_prompt["messages"][0]["content"] = (
        current_pattern_prompt["messages"][0]["content"]
        .replace("{engine_name}", engine_name)
        .replace("{repo_url}", engine["repo_url"])
        .replace("{doc_url}", engine["doc_url"])
    )
    current_pattern_prompt["messages"][1]["content"] = (
        current_pattern_prompt["messages"][1]["content"]
        .replace("{engine_name}", engine_name)
    )

    pattern_agent_config = {
        "configurable": {"thread_id": "pattern_mining_{}".format(engine["short_name"])}
    }
    pattern_response = operator_retrieval_agent.invoke(
        input={"messages": current_pattern_prompt["messages"]},
        config=pattern_agent_config,
    )
    pattern_text = pattern_response["messages"][-1].content

    # Parse the JSON list of patterns
    try:
        json_start = pattern_text.find("[")
        json_end = pattern_text.rfind("]") + 1
        if json_start != -1 and json_end > json_start:
            patterns = json.loads(pattern_text[json_start:json_end])
        else:
            patterns = json.loads(pattern_text)
    except json.JSONDecodeError:
        print("Warning: Failed to parse patterns for {}. Retrying...".format(engine_name))
        retry_response = operator_retrieval_agent.invoke(
            input={"messages": [
                {"role": "user", "content": (
                    "The previous response could not be parsed as JSON. Please output ONLY "
                    "a valid JSON list of optimizable patterns for {}. Format: "
                    '[{{"pattern_name": "...", "operator_sequence": ["Op1", "Op2"], '
                    '"optimization_type": "fusion", "description": "..."}}, ...]. No extra text.'
                ).format(engine_name)}
            ]},
            config=pattern_agent_config,
        )
        retry_text = retry_response["messages"][-1].content
        json_start = retry_text.find("[")
        json_end = retry_text.rfind("]") + 1
        patterns = json.loads(retry_text[json_start:json_end])

    engine_optimizable_patterns[engine_name] = patterns
    print("Found {} optimizable patterns for {}:".format(len(patterns), engine_name))
    for p in patterns:
        print("  - {} ({}): {} -> {}".format(
            p["pattern_name"], p["optimization_type"],
            " + ".join(p["operator_sequence"]), p["description"][:60]
        ))

    # Store patterns in the knowledge database
    for pattern in patterns:
        operator_knowledge_db.add_documents(
            documents=[Document(
                page_content="Engine: {} | Pattern: {} | Operators: {} | Type: {} | {}".format(
                    engine_name, pattern["pattern_name"],
                    " -> ".join(pattern["operator_sequence"]),
                    pattern["optimization_type"], pattern["description"],
                ),
                metadata={
                    "engine": engine_name,
                    "pattern": pattern["pattern_name"],
                    "type": "optimizable_pattern",
                },
            )]
        )

In [ ]:
# Stage 2: Model Reorganization (§3.2, Eq. 6)
# Clone the model into k copies, one per engine.
# For each copy, apply engine-specific optimizations by rewriting code via P.3.
# M -> {m_1, m_2, ..., m_k}

# First, detect which patterns from each engine actually exist in the model
def find_pattern_instances_in_model(operators, pattern_sequence):
    """Find all instances of a pattern (operator sequence) in the model's operator list.
    Returns list of (start_index, end_index) tuples."""
    instances = []
    pattern_len = len(pattern_sequence)
    for i in range(len(operators) - pattern_len + 1):
        match = True
        for p in range(pattern_len):
            if operators[i + p]["op_type"] != pattern_sequence[p]:
                match = False
                break
        if match:
            instances.append((i, i + pattern_len - 1))
    return instances


# Build pattern instance map for each engine
# Records which patterns exist in the model and their locations
engine_pattern_instances = {}  # engine_name -> list of {pattern, instances}

for engine in available_engines:
    engine_name = engine["name"]
    patterns = engine_optimizable_patterns.get(engine_name, [])
    pattern_instances = []

    for pattern in patterns:
        op_seq = pattern["operator_sequence"]
        instances = find_pattern_instances_in_model(model_operators, op_seq)
        if instances:
            pattern_instances.append({
                "pattern": pattern,
                "instances": instances,
            })

    engine_pattern_instances[engine_name] = pattern_instances
    if pattern_instances:
        print("{}: Found {} applicable patterns with {} total instances in the model.".format(
            engine_name,
            len(pattern_instances),
            sum(len(pi["instances"]) for pi in pattern_instances),
        ))
    else:
        print("{}: No applicable patterns found in the model.".format(engine_name))

# Generate the reorganized model code for each engine via P.3
# The agent rewrites engine-specific optimization code into standalone functions
reorganized_model_code = {}  # engine_name -> list of optimization function code strings

for engine in available_engines:
    engine_name = engine["name"]
    pattern_instances = engine_pattern_instances.get(engine_name, [])

    if not pattern_instances:
        reorganized_model_code[engine_name] = []
        continue

    print("\n=== Generating reorganization code for {} ===".format(engine_name))
    engine_reorg_code = []

    for pi in pattern_instances:
        pattern = pi["pattern"]
        pattern_name = pattern["pattern_name"].replace(" ", "_").replace("-", "_").lower()

        current_rewrite_prompt = copy.deepcopy(pattern_code_rewriting_prompt)
        current_rewrite_prompt["messages"][0]["content"] = (
            current_rewrite_prompt["messages"][0]["content"]
            .replace("{pattern_info}", json.dumps(pattern, indent=2))
            .replace("{engine_name}", engine_name)
        )
        current_rewrite_prompt["messages"][1]["content"] = (
            current_rewrite_prompt["messages"][1]["content"]
            .replace("{pattern_name}", pattern_name)
        )

        rewrite_agent_config = {
            "configurable": {"thread_id": "rewrite_{}_{}".format(engine["short_name"], pattern_name)}
        }
        rewrite_response = operator_retrieval_agent.invoke(
            input={"messages": current_rewrite_prompt["messages"]},
            config=rewrite_agent_config,
        )
        rewrite_code = rewrite_response["messages"][-1].content
        engine_reorg_code.append({
            "pattern_name": pattern_name,
            "code": rewrite_code,
            "instances": pi["instances"],
        })
        print("  Generated optimization function: apply_{}".format(pattern_name))

    reorganized_model_code[engine_name] = engine_reorg_code

# Create reorganized model copies by applying the optimization functions
# m_j ~ {o_j_1, ..., {o_j_r, o_j_r+1, ...} -> o_j_r_hat, ...}
reorganized_models = {}  # engine_name -> reorganized ONNX model

for engine in available_engines:
    engine_name = engine["name"]
    reorg_code_list = reorganized_model_code.get(engine_name, [])

    # Start with a clone of the original model
    model_copy = copy.deepcopy(onnx_model)

    if reorg_code_list:
        # Execute each optimization function on the model copy
        for reorg_entry in reorg_code_list:
            code_text = reorg_entry["code"]
            # Extract the Python code from the response
            sanitized_code = tool_list.sanitize_code(code_text)
            if sanitized_code:
                try:
                    # Execute the optimization function in a controlled namespace
                    exec_namespace = {"onnx": onnx, "numpy": np, "copy": copy}
                    exec(sanitized_code, exec_namespace)

                    # Find and call the optimization function
                    func_name = "apply_{}".format(reorg_entry["pattern_name"])
                    if func_name in exec_namespace:
                        model_copy = exec_namespace[func_name](model_copy)
                        print("Applied {} to {} model copy.".format(func_name, engine_name))
                    else:
                        # Try to find any function that was defined
                        for key, val in exec_namespace.items():
                            if callable(val) and key.startswith("apply_"):
                                model_copy = val(model_copy)
                                print("Applied {} to {} model copy.".format(key, engine_name))
                                break
                except Exception as e:
                    print("Warning: Failed to apply {} for {}: {}".format(
                        reorg_entry["pattern_name"], engine_name, str(e)
                    ))

    reorganized_models[engine_name] = model_copy

# Count operators in each reorganized model
for engine_name, reorg_model in reorganized_models.items():
    reorg_op_count = len(list(reorg_model.graph.node))
    print("{}: {} operators after reorganization (original: {})".format(
        engine_name, reorg_op_count, num_operators
    ))

In [ ]:
# =================== Technical Module 3: Greedy Cross-Engine Inference Algorithm (§3.3) ===================
# This module partitions the model and assigns each segment to the optimal engine
# by jointly considering: operator-engine compatibility, optimizable patterns,
# and engine-switching overhead.

# Step 1: Profile operator execution times T[n x k] and engine switching times S[k x k]
# Using nn-Meter-style profiling via the agent on the target device.

operator_profiling_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are an expert in AI model profiling on embedded devices. Your task is to "
                "generate Python code that profiles the execution time of individual ONNX operators "
                "on different inference engines.\n\n"
                "The code should:\n"
                "1. Load an ONNX model and extract individual operators.\n"
                "2. For each operator, create a minimal sub-graph containing only that operator.\n"
                "3. Run the sub-graph on each available inference engine and measure execution time.\n"
                "4. Also measure the context switching overhead between engines.\n"
                "5. Output results as JSON with 'operator_times' (dict of op_name -> dict of engine -> time_ms) "
                "and 'switching_times' (dict of engine_pair -> time_ms).\n\n"
                "Available engines: {engine_list}\n"
                "Model path: {model_path}\n"
                "Target device: {device_info}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Please generate a Python profiling script that measures the execution time of "
                "each operator in the ONNX model on each available inference engine, as well as "
                "the context switching time between engines. Output only the Python code."
            ),
        },
    ]
}

# Generate profiling code using the agent
engine_list_str = ", ".join(e["name"] for e in available_engines)
current_profiling_prompt = copy.deepcopy(operator_profiling_prompt)
current_profiling_prompt["messages"][0]["content"] = (
    current_profiling_prompt["messages"][0]["content"]
    .replace("{engine_list}", engine_list_str)
    .replace("{model_path}", optimized_model_path)
    .replace("{device_info}", "Raspberry Pi (CPU runtime)")
)

profiling_agent_config = {
    "configurable": {"thread_id": "operator_profiling"}
}
profiling_response = operator_retrieval_agent.invoke(
    input={"messages": current_profiling_prompt["messages"]},
    config=profiling_agent_config,
)
profiling_code = profiling_response["messages"][-1].content

# Save and execute the profiling script on the target device
profiling_script = tool_list.sanitize_code(profiling_code)
with open("operator_profiling.py", "w", encoding="utf-8") as f:
    f.write(profiling_script)

# Transfer the profiling script and model to the target device for execution
cross_platform_protocol.send_file(
    file_path=optimized_model_path,
    url="{}:{}/model_transmission".format(config.server_address, config.port),
    meta_information={
        "model_name": model_name,
        "model_weight_name": os.path.basename(optimized_model_path),
        "model_size": os.path.getsize(optimized_model_path),
    },
)
cross_platform_protocol.send_file(
    file_path="operator_profiling.py",
    url="{}:{}/script_transmission".format(config.server_address, config.port),
    meta_information={"script_name": "operator_profiling.py"},
)

# Execute the profiling script on the target device
profiling_result = cross_platform_protocol.send_command(
    url="{}:{}/model_execution".format(config.server_address, config.port),
    meta_information={
        "command": "python3 {} --model_weight_path {} --dataset_path {} > log.txt",
        "model_weight_name": os.path.basename(optimized_model_path),
        "script_name": "operator_profiling.py",
    },
)

# Parse profiling results
# If profiling execution fails or returns incomplete data, use estimated times
try:
    profiling_data = json.loads(profiling_result)
    operator_exec_times = profiling_data.get("operator_times", {})
    switching_times = profiling_data.get("switching_times", {})
except (json.JSONDecodeError, TypeError):
    print("Warning: Could not parse profiling results. Using LLM-estimated execution times.")
    # Use the agent to estimate execution times based on operator types and engine characteristics
    estimation_response = tool_list.model.invoke(
        [{"role": "user", "content": (
            "For the following ONNX operators, estimate their relative execution times (in ms) "
            "on each inference engine ({}) running on a Raspberry Pi (CPU mode). "
            "Also estimate context switching overhead between each pair of engines. "
            "Operators: {}. Output as JSON with 'operator_times' and 'switching_times'."
        ).format(engine_list_str, list(set(op["op_type"] for op in model_operators)))}]
    )
    est_text = estimation_response.content
    try:
        json_start = est_text.find("{")
        json_end = est_text.rfind("}") + 1
        profiling_data = json.loads(est_text[json_start:json_end])
        operator_exec_times = profiling_data.get("operator_times", {})
        switching_times = profiling_data.get("switching_times", {})
    except (json.JSONDecodeError, ValueError):
        operator_exec_times = {}
        switching_times = {}

# Build execution time matrix T[n x k] and switching time matrix S[k x k]
T = np.ones((num_operators, num_engines)) * 1.0  # default: 1.0ms per operator
S = np.ones((num_engines, num_engines)) * 0.5     # default: 0.5ms switching overhead
np.fill_diagonal(S, 0.0)  # no cost for staying on the same engine

# Fill T matrix with profiled/estimated values
for i, op in enumerate(model_operators):
    op_type = op["op_type"]
    if op_type in operator_exec_times:
        for j, engine in enumerate(available_engines):
            engine_name = engine["name"]
            if engine_name in operator_exec_times[op_type]:
                T[i, j] = operator_exec_times[op_type][engine_name]

# Fill S matrix with switching times
for j1, e1 in enumerate(available_engines):
    for j2, e2 in enumerate(available_engines):
        if j1 != j2:
            key = "{}_to_{}".format(e1["short_name"], e2["short_name"])
            if key in switching_times:
                S[j1, j2] = switching_times[key]

print("Operator execution time matrix T shape: {}".format(T.shape))
print("Engine switching time matrix S shape: {}".format(S.shape))

In [ ]:
# Step 2: Greedy Cross-Engine Inference Algorithm (Eq. 7-9)
# The algorithm sequentially assigns each operator to the engine that minimizes
# the marginal cost: MC(i,j) = T[i,j] - λ_P * ΔP_hat(i,j) + λ_S * 1[e_j ≠ e_c] * S[c,j]

# Hyperparameters
LAMBDA_P = 1.0       # weight for pattern credit term
LAMBDA_S = 1.0       # weight for switching penalty term
ALPHA = 0.1          # discount factor for pattern credit
LOOKAHEAD_W = 5      # lookahead window size for pattern completions
BACKTRACK_B = 3      # bounded backtracking depth
REGRET_RHO = 0.2     # regret threshold for backtracking (fraction of recent window cost)


def compute_pattern_saving_credit(op_index, engine_idx, model_ops, compatibility,
                                  engine_patterns, exec_times, lookahead_w):
    """Compute the discounted pattern saving credit ΔP_hat(i, j) (Eq. 7-8).
    
    For each optimizable pattern that completes within [i, i+w-1] when kept on engine e_j,
    compute the time saved by the optimization versus sequential execution.
    """
    engine_name = available_engines[engine_idx]["name"]
    patterns = engine_pattern_instances.get(engine_name, [])
    total_credit = 0.0

    for pi in patterns:
        pattern = pi["pattern"]
        op_seq = pattern["operator_sequence"]
        pattern_len = len(op_seq)

        for (start, end) in pi["instances"]:
            # Check if this pattern completes within the lookahead window from op_index
            if start <= op_index <= end and end < op_index + lookahead_w:
                # Check if all operators in this pattern are compatible with this engine
                all_compatible = True
                for t in range(start, end + 1):
                    if compatibility[t, engine_idx] != 1:
                        all_compatible = False
                        break

                if all_compatible:
                    # Δ(p, j) = sum of T[r, j] for r in pattern - T_optimized
                    sequential_time = sum(exec_times[r, engine_idx] for r in range(start, end + 1))
                    # Optimized time: assume fusion reduces to ~60% of the dominant operator
                    optimized_time = max(exec_times[r, engine_idx] for r in range(start, end + 1)) * 0.6

                    delta = sequential_time - optimized_time

                    # Apply discount γ(p, i, j) = Π * exp(-α * X_hat)
                    # X_hat estimates extra switches needed to keep span on e_j
                    discount = np.exp(-ALPHA * abs(end - op_index))
                    total_credit += discount * delta

    return total_credit


def greedy_cross_engine_partition(model_ops, compatibility, exec_times, switch_times,
                                  lambda_p, lambda_s, lookahead_w, backtrack_b, regret_rho):
    """Greedy Cross-Engine Inference Algorithm with bounded backtracking (§3.3).
    
    Returns:
        assignments: list of engine indices, one per operator
        partitions: list of (start_idx, end_idx, engine_idx) tuples
    """
    n = len(model_ops)
    k = compatibility.shape[1]
    assignments = [-1] * n  # engine index for each operator
    history = []  # (op_index, engine_idx, marginal_cost) for backtracking

    # Assign the first operator to the engine with the shortest execution time
    # among compatible engines
    compatible_first = [j for j in range(k) if compatibility[0, j] == 1]
    if compatible_first:
        best_first = min(compatible_first, key=lambda j: exec_times[0, j])
        assignments[0] = best_first
    else:
        # Fallback to ONNX Runtime (index 0) - CPU runtime as last resort
        assignments[0] = 0
    
    current_engine = assignments[0]
    history.append((0, current_engine, exec_times[0, current_engine]))

    # Greedily assign each subsequent operator
    for i in range(1, n):
        compatible_engines = [j for j in range(k) if compatibility[i, j] == 1]

        if not compatible_engines:
            # No compatible engine: fallback to ONNX Runtime (CPU)
            assignments[i] = 0
            current_engine = 0
            history.append((i, 0, exec_times[i, 0]))
            continue

        # Compute marginal cost for each compatible engine (Eq. 9)
        best_engine = None
        best_cost = float("inf")

        for j in compatible_engines:
            # Latency term: T[i, j]
            latency = exec_times[i, j]

            # Pattern credit term: ΔP_hat(i, j)
            pattern_credit = compute_pattern_saving_credit(
                i, j, model_ops, compatibility,
                engine_optimizable_patterns, exec_times, lookahead_w,
            )

            # Switch penalty: λ_S * 1[e_j ≠ e_c] * S[c, j]
            switch_penalty = 0.0
            if j != current_engine:
                switch_penalty = switch_times[current_engine, j]

            # MC(i, j) = T[i,j] - λ_P * ΔP_hat(i,j) + λ_S * 1[e_j ≠ e_c] * S[c,j]
            marginal_cost = latency - lambda_p * pattern_credit + lambda_s * switch_penalty

            # Tie-breaking: favor the current engine to minimize switching
            if marginal_cost < best_cost or (marginal_cost == best_cost and j == current_engine):
                best_cost = marginal_cost
                best_engine = j

        assignments[i] = best_engine
        history.append((i, best_engine, best_cost))

        # Bounded Backtracking: check if reverting recent decisions could unlock
        # a high-value pattern
        if len(history) > 1 and best_engine != current_engine:
            # Look back at most B steps
            backtrack_start = max(0, len(history) - backtrack_b)
            recent_cost = sum(h[2] for h in history[backtrack_start:])

            for t_idx in range(backtrack_start, len(history) - 1):
                t_op_idx = history[t_idx][0]
                t_engine = history[t_idx][1]

                # Check if switching this past decision could complete a pattern
                for alt_j in range(k):
                    if alt_j == t_engine or compatibility[t_op_idx, alt_j] != 1:
                        continue

                    alt_credit = compute_pattern_saving_credit(
                        t_op_idx, alt_j, model_ops, compatibility,
                        engine_optimizable_patterns, exec_times, lookahead_w,
                    )

                    # If potential savings exceed regret threshold
                    if alt_credit > regret_rho * recent_cost:
                        # Revert: replay from t_idx forward with the alternative engine
                        assignments[t_op_idx] = alt_j
                        replay_engine = alt_j

                        for replay_i in range(t_op_idx + 1, i + 1):
                            replay_compatible = [jj for jj in range(k) if compatibility[replay_i, jj] == 1]
                            if not replay_compatible:
                                assignments[replay_i] = 0
                                replay_engine = 0
                                continue

                            replay_best = None
                            replay_best_cost = float("inf")
                            for jj in replay_compatible:
                                r_latency = exec_times[replay_i, jj]
                                r_credit = compute_pattern_saving_credit(
                                    replay_i, jj, model_ops, compatibility,
                                    engine_optimizable_patterns, exec_times, lookahead_w,
                                )
                                r_switch = switch_times[replay_engine, jj] if jj != replay_engine else 0.0
                                r_mc = r_latency - lambda_p * r_credit + lambda_s * r_switch
                                if r_mc < replay_best_cost or (r_mc == replay_best_cost and jj == replay_engine):
                                    replay_best_cost = r_mc
                                    replay_best = jj

                            assignments[replay_i] = replay_best
                            replay_engine = replay_best

                        # Update history after backtracking
                        history = history[:backtrack_start]
                        for h_i in range(t_op_idx, i + 1):
                            history.append((h_i, assignments[h_i], exec_times[h_i, assignments[h_i]]))
                        break
                else:
                    continue
                break

        current_engine = assignments[i]

    # Convert assignments to partitions (consecutive operators on the same engine)
    partitions = []
    if assignments:
        seg_start = 0
        seg_engine = assignments[0]
        for i in range(1, n):
            if assignments[i] != seg_engine:
                partitions.append((seg_start, i - 1, seg_engine))
                seg_start = i
                seg_engine = assignments[i]
        partitions.append((seg_start, n - 1, seg_engine))

    return assignments, partitions


# Run the greedy algorithm
operator_assignments, model_partitions = greedy_cross_engine_partition(
    model_operators, compatibility_matrix, T, S,
    LAMBDA_P, LAMBDA_S, LOOKAHEAD_W, BACKTRACK_B, REGRET_RHO,
)

# Print partitioning results
print("\n=== Model Partitioning Results ===")
print("Total partitions: {}".format(len(model_partitions)))
for idx, (start, end, engine_idx) in enumerate(model_partitions):
    engine_name = available_engines[engine_idx]["name"]
    op_count = end - start + 1
    op_types = [model_operators[i]["op_type"] for i in range(start, end + 1)]
    print("Partition {}: operators [{}-{}] ({} ops) -> {} | Types: {}".format(
        idx, start, end, op_count, engine_name,
        list(set(op_types))[:5],  # show first 5 unique types
    ))

# Compute estimated total inference time
total_latency = 0.0
for start, end, engine_idx in model_partitions:
    for i in range(start, end + 1):
        total_latency += T[i, engine_idx]

# Add switching overhead
for idx in range(1, len(model_partitions)):
    prev_engine = model_partitions[idx - 1][2]
    curr_engine = model_partitions[idx][2]
    if prev_engine != curr_engine:
        total_latency += S[prev_engine, curr_engine]

print("\nEstimated total inference latency: {:.2f} ms".format(total_latency))
print("Number of engine switches: {}".format(
    sum(1 for idx in range(1, len(model_partitions))
        if model_partitions[idx][2] != model_partitions[idx - 1][2])
))

In [ ]:
# Step 3: Generate cross-engine inference execution code
# Extract ONNX sub-models for each partition and generate the deployment script
# that alternately executes partitions across different engines.

cross_engine_execution_prompt = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are an expert in deploying AI models on embedded devices with multiple "
                "inference engines. Your task is to generate a Python script that performs "
                "cross-engine model inference by executing different model partitions on "
                "different inference engines and passing intermediate results between them.\n\n"
                "The script should:\n"
                "1. Load each model partition (sub-model) as a separate ONNX model.\n"
                "2. Create inference sessions for each partition using the assigned engine.\n"
                "3. Execute partitions sequentially, passing outputs from one as inputs to the next.\n"
                "4. Measure and report total inference latency.\n"
                "5. Support command-line arguments for model paths and dataset path.\n\n"
                "Model partitions:\n{partition_info}\n\n"
                "Available engines and their Python APIs:\n"
                "- ONNX Runtime: onnxruntime.InferenceSession with providers=['CPUExecutionProvider']\n"
                "- OpenVINO: openvino.runtime.Core().compile_model()\n"
                "- TensorRT: tensorrt engine via onnx-tensorrt or trtexec\n\n"
                "Custom operator registration code (if any):\n{custom_op_code}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Please generate a complete Python script for cross-engine inference of the "
                "partitioned model. The script should handle data transfer between engines "
                "and measure total inference time. Output only the Python code."
            ),
        },
    ]
}

# Prepare partition info string
partition_info_str = ""
for idx, (start, end, engine_idx) in enumerate(model_partitions):
    engine_name = available_engines[engine_idx]["name"]
    op_types = [model_operators[i]["op_type"] for i in range(start, end + 1)]
    partition_info_str += "Partition {}: operators [{}-{}] -> {} (ops: {})\n".format(
        idx, start, end, engine_name, ", ".join(set(op_types))
    )

# Prepare custom op registration code string
custom_op_code_str = ""
for op_type, engine_codes in custom_operator_code.items():
    for engine_name, code in engine_codes.items():
        custom_op_code_str += "\n# Custom registration for {} on {}:\n{}\n".format(
            op_type, engine_name, code
        )
if not custom_op_code_str:
    custom_op_code_str = "None required."

# Generate the cross-engine execution script
current_exec_prompt = copy.deepcopy(cross_engine_execution_prompt)
current_exec_prompt["messages"][0]["content"] = (
    current_exec_prompt["messages"][0]["content"]
    .replace("{partition_info}", partition_info_str)
    .replace("{custom_op_code}", custom_op_code_str)
)

exec_agent_config = {
    "configurable": {"thread_id": "cross_engine_execution"}
}
exec_response = operator_retrieval_agent.invoke(
    input={"messages": current_exec_prompt["messages"]},
    config=exec_agent_config,
)
cross_engine_script = exec_response["messages"][-1].content

# Also extract the sub-models for each partition from the ONNX model
import onnx
from onnx import helper, TensorProto

def extract_submodel(original_model, start_idx, end_idx, partition_idx):
    """Extract a sub-model from the original ONNX model containing operators [start_idx, end_idx]."""
    graph = original_model.graph
    nodes = list(graph.node)[start_idx:end_idx + 1]

    # Collect all inputs and outputs for the sub-graph
    all_inputs = set()
    all_outputs = set()
    internal_tensors = set()

    for node in nodes:
        for inp in node.input:
            all_inputs.add(inp)
        for out in node.output:
            all_outputs.add(out)
            internal_tensors.add(out)

    # Sub-graph inputs: inputs that are not produced by nodes in the sub-graph
    sub_inputs = all_inputs - internal_tensors
    # Sub-graph outputs: outputs that are consumed by nodes outside the sub-graph or are final outputs
    external_outputs = set()
    all_graph_nodes = list(graph.node)
    for i, node in enumerate(all_graph_nodes):
        if i < start_idx or i > end_idx:
            for inp in node.input:
                if inp in all_outputs:
                    external_outputs.add(inp)

    # Also include final graph outputs
    for out in graph.output:
        if out.name in all_outputs:
            external_outputs.add(out.name)

    # If no external outputs found, use all outputs of the last node
    if not external_outputs:
        external_outputs = set(nodes[-1].output)

    # Build input/output value info
    # Collect existing value_info and initializers
    value_info_map = {}
    for vi in list(graph.input) + list(graph.output) + list(graph.value_info):
        value_info_map[vi.name] = vi

    sub_input_vis = []
    for name in sub_inputs:
        if name in value_info_map:
            sub_input_vis.append(value_info_map[name])
        elif name:  # skip empty string inputs
            # Create a placeholder value_info
            sub_input_vis.append(helper.make_tensor_value_info(name, TensorProto.FLOAT, None))

    sub_output_vis = []
    for name in external_outputs:
        if name in value_info_map:
            sub_output_vis.append(value_info_map[name])
        elif name:
            sub_output_vis.append(helper.make_tensor_value_info(name, TensorProto.FLOAT, None))

    # Collect relevant initializers
    initializer_map = {init.name: init for init in graph.initializer}
    sub_initializers = []
    for name in sub_inputs:
        if name in initializer_map:
            sub_initializers.append(initializer_map[name])

    # Create the sub-graph and sub-model
    sub_graph = helper.make_graph(
        nodes, "partition_{}".format(partition_idx),
        sub_input_vis, sub_output_vis, sub_initializers,
    )
    sub_model = helper.make_model(sub_graph)
    sub_model.ir_version = original_model.ir_version
    for opset in original_model.opset_import:
        sub_opset = sub_model.opset_import.add()
        sub_opset.domain = opset.domain
        sub_opset.version = opset.version

    return sub_model


# Extract and save sub-models for each partition
partition_model_paths = []
for idx, (start, end, engine_idx) in enumerate(model_partitions):
    sub_model_path = "partition_{}.onnx".format(idx)
    try:
        sub_model = extract_submodel(onnx_model, start, end, idx)
        onnx.save(sub_model, sub_model_path)
        partition_model_paths.append(sub_model_path)
        print("Saved partition {} ({} ops, engine: {}) to {}".format(
            idx, end - start + 1, available_engines[engine_idx]["name"], sub_model_path
        ))
    except Exception as e:
        print("Warning: Failed to extract partition {}: {}".format(idx, str(e)))
        partition_model_paths.append(None)

print("\nExtracted {} partition sub-models.".format(len(partition_model_paths)))

In [ ]:
# Step 4: Deploy partitioned model to the target embedded device and execute cross-engine inference
# Transfer all partition sub-models and the cross-engine execution script to the device.

# Save the cross-engine execution script
cross_engine_script_sanitized = tool_list.sanitize_code(cross_engine_script)
with open("cross_engine_inference.py", "w", encoding="utf-8") as f:
    f.write(cross_engine_script_sanitized)

# Transfer each partition sub-model to the target device
for idx, sub_model_path in enumerate(partition_model_paths):
    if sub_model_path and os.path.exists(sub_model_path):
        cross_platform_protocol.send_file(
            file_path=sub_model_path,
            url="{}:{}/model_transmission".format(config.server_address, config.port),
            meta_information={
                "model_name": "{}_partition_{}".format(model_name, idx),
                "model_weight_name": sub_model_path,
                "model_size": os.path.getsize(sub_model_path),
            },
        )
        print("Transferred {} to target device.".format(sub_model_path))

# Also transfer the full optimized model (needed for engines that optimize the whole graph)
cross_platform_protocol.send_file(
    file_path=optimized_model_path,
    url="{}:{}/model_transmission".format(config.server_address, config.port),
    meta_information={
        "model_name": model_name,
        "model_weight_name": os.path.basename(optimized_model_path),
        "model_size": os.path.getsize(optimized_model_path),
    },
)

# Transfer the cross-engine inference script
cross_platform_protocol.send_file(
    file_path="cross_engine_inference.py",
    url="{}:{}/script_transmission".format(config.server_address, config.port),
    meta_information={"script_name": "cross_engine_inference.py"},
)

# Save the partition configuration as JSON for the execution script
partition_config = {
    "model_name": model_name,
    "original_model": os.path.basename(optimized_model_path),
    "partitions": [
        {
            "index": idx,
            "start_op": start,
            "end_op": end,
            "engine": available_engines[engine_idx]["name"],
            "engine_short": available_engines[engine_idx]["short_name"],
            "sub_model": "partition_{}.onnx".format(idx),
            "num_operators": end - start + 1,
        }
        for idx, (start, end, engine_idx) in enumerate(model_partitions)
    ],
    "num_partitions": len(model_partitions),
    "num_engine_switches": sum(
        1 for idx in range(1, len(model_partitions))
        if model_partitions[idx][2] != model_partitions[idx - 1][2]
    ),
    "custom_operators": list(custom_operator_code.keys()),
}

with open("partition_config.json", "w", encoding="utf-8") as f:
    json.dump(partition_config, f, indent=2)

# Transfer the partition config
cross_platform_protocol.send_file(
    file_path="partition_config.json",
    url="{}:{}/script_transmission".format(config.server_address, config.port),
    meta_information={"script_name": "partition_config.json"},
)

print("\nAll files transferred to the target device.")

In [ ]:
# Step 5: Execute cross-engine inference on the target device and collect results

# Execute the cross-engine inference script on the target device
execution_result = cross_platform_protocol.send_command(
    url="{}:{}/model_execution".format(config.server_address, config.port),
    meta_information={
        "command": "python3 {} --model_weight_path {} --dataset_path {} > log.txt",
        "model_weight_name": os.path.basename(optimized_model_path),
        "script_name": "cross_engine_inference.py",
    },
)

# Filter the execution output to extract performance metrics
edge_model_monitoring_output_filtering_chain = (
    prompt.model_monitoring_output_filtering_prompt | tool_list.model
)
filtered_program_output = edge_model_monitoring_output_filtering_chain.invoke(
    execution_result
)
filtered_program_output = filtered_program_output.content

print("\n=== Cross-Engine Inference Results ===")
print(filtered_program_output)

# Judge whether the model performance meets the requirements
performance_requirement_judgement_chain = (
    prompt.performance_requirement_judgement_prompt | tool_list.model
)
judgement_result = performance_requirement_judgement_chain.invoke(
    input={
        "performance_requirement": performance_requirement,
        "performance_monitoring_result": filtered_program_output,
    }
)
judgement_result = judgement_result.content

print("\n=== Requirement Satisfaction ===")
if judgement_result.lower() == "yes":
    print("The cross-engine deployed model meets ALL performance requirements.")
else:
    print("The cross-engine deployed model does NOT meet all performance requirements.")
    print("Consider adjusting the greedy algorithm parameters (λ_P, λ_S, α) or trying different engine configurations.")

# Print final deployment summary
print("\n=== AutoDP Deployment Summary ===")
print("Model: {}".format(model_name))
print("Optimized model: {}".format(optimized_model_path))
print("Number of operators: {}".format(num_operators))
print("Number of partitions: {}".format(len(model_partitions)))
print("Number of engine switches: {}".format(partition_config["num_engine_switches"]))
print("Engines used: {}".format(
    list(set(available_engines[engine_idx]["name"] for _, _, engine_idx in model_partitions))
))
print("Custom operators registered: {}".format(
    list(custom_operator_code.keys()) if custom_operator_code else "None"
))
print("\nAutoDP deployment complete.")